# Boundary Case Study Across Environments

This notebook studies **sentence-boundary changes in localized deception rate** across five environments:

- `bs`
- `interview`
- `advisor_audit`
- `car_sales`
- `gridworld`

The analysis is intentionally driven by a **small set of env-specific token groups** that capture the core private or strategic information needed to deceive.


## 1. Imports and configuration

Each environment points at its own DeepSeek-localized run, but the analysis surface stays the same.


In [7]:
import ast
import base64
import gc
import html
import importlib
import json
import math
import os
import re
import sys
import warnings
from io import BytesIO
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.environ['CUDA_VISIBLE_DEVICES'] = '7'

try:
    import torch
except Exception as exc:
    raise ImportError('PyTorch is required for this notebook.') from exc

try:
    from IPython.display import HTML, Image as IPythonImage, Markdown, clear_output, display
except Exception:
    HTML = None
    IPythonImage = None
    Markdown = None

    def clear_output(wait=False):
        return None

    def display(value):
        print(value)

NOTEBOOK_ROOT = Path('/playpen-ssd/smerrill/deception2/Notebooks')
SRC_ROOT = Path('/playpen-ssd/smerrill/deception2/src')
for root in [NOTEBOOK_ROOT, SRC_ROOT]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

import bs_gpt_oss_probe_commitment_lib as bpc
import attention_features2 as af2
from attention_features import (
    add_span_match_columns,
    align_localized_sentences_to_tokens,
    build_localized_sentence_df,
    token_indices_for_char_span,
)

bpc = importlib.reload(bpc)
af2 = importlib.reload(af2)

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 240)
torch.set_grad_enabled(False)
plt.ioff()

OUTPUT_ROOT = NOTEBOOK_ROOT / 'boundary_case_study_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_DEVICE = 'cuda'
MODEL_DTYPE = 'bfloat16'
MODEL_TRUST_REMOTE_CODE = True
SHARED_MODEL_NAME = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'

BOUNDARY_REQUIRE_PREFIX_GAP_ONE = True
BOUNDARY_REQUIRE_SENTENCE_GAP_ONE = True
FORCE_REBUILD_BOUNDARY_CACHE = False

LARGE_SPIKE_DELTA_THRESHOLD = 0.20
FLAT_MAX_DELTA_THRESHOLD = 0.05
EXAMPLE_WIDGET_LIMIT = 300

DEFAULT_QUERY_MODE = 'last_token_selected_sentence'
TOP_HEADS_TO_SHOW = 16
TOP_DIMENSIONS_TO_SHOW = 20
GROUP_PREVIEW_CHARS = 160

ENV_ORDER = ['bs', 'interview', 'advisor_audit', 'car_sales', 'gridworld']
DEFAULT_ENV = 'bs'
SENTENCE_OFFSET_GROUP_ORDER = [
    'sentence_i_minus_1',
    'sentence_i_minus_2',
    'sentence_i_minus_3',
    'sentence_i_minus_4',
    'recent_prior_sentences',
    'earlier_prior_sentences',
    'all_prior_sentences',
    'selected_sentence',
]
ATTENTION_GROUP_LABELS = {
    'sentence_i_minus_1': 'sentence i-1',
    'sentence_i_minus_2': 'sentence i-2',
    'sentence_i_minus_3': 'sentence i-3',
    'sentence_i_minus_4': 'sentence i-4',
    'recent_prior_sentences': 'recent prior sentences',
    'earlier_prior_sentences': 'earlier prior sentences',
    'all_prior_sentences': 'all prior sentences',
    'selected_sentence': 'sentence i',
}
SENTENCE_OFFSET_PANELS = [
    ('Nearest prior sentences', ['sentence_i_minus_1', 'sentence_i_minus_2', 'sentence_i_minus_3', 'sentence_i_minus_4']),
    ('Broader prior context', ['recent_prior_sentences', 'earlier_prior_sentences', 'all_prior_sentences', 'selected_sentence']),
]
FEATURE_RECENT_WINDOW_TOKENS = int(getattr(af2, 'DEFAULT_RECENT_WINDOW_TOKENS', 64))

def env_output_dir(env_name):
    path = OUTPUT_ROOT / env_name
    path.mkdir(parents=True, exist_ok=True)
    return path

def _env_dataset_paths(env_name):
    model_root = Path(f'/playpen-ssd/smerrill/deception2/DatasetMain/{env_name}/DeepSeek-R1-Distill-Qwen-7B')
    return {
        'dataset_root': model_root,
        'localization_dir': model_root / 'localization',
    }

def _build_env_config(env_name):
    dataset_paths = _env_dataset_paths(env_name)
    return {
        'dataset_root': dataset_paths['dataset_root'],
        'localization_dir': dataset_paths['localization_dir'],
        'boundary_cache_path': env_output_dir(env_name) / 'boundary_catalog_raw.csv',
        'model_name': SHARED_MODEL_NAME,
        'num_layers': 28,
        'default_focus_group': 'selected_sentence',
        'attention_group_order': list(SENTENCE_OFFSET_GROUP_ORDER),
        'residual_target_group_order': ['sentence_i_minus_1', 'sentence_i_minus_2', 'sentence_i_minus_3', 'sentence_i_minus_4', 'all_prior_sentences'],
        'attention_panels': list(SENTENCE_OFFSET_PANELS),
    }

ENV_CONFIGS = {env_name: _build_env_config(env_name) for env_name in ENV_ORDER}

DEFAULT_FOCUS_GROUP = ENV_CONFIGS[DEFAULT_ENV]['default_focus_group']

config_rows = []
for env_name in ENV_ORDER:
    cfg = ENV_CONFIGS[env_name]
    config_rows.append(
        {
            'env': env_name,
            'dataset_root': str(cfg['dataset_root']),
            'localization_dir': str(cfg['localization_dir']),
            'boundary_cache_path': str(cfg['boundary_cache_path']),
            'model_name': cfg['model_name'],
            'num_layers': int(cfg['num_layers']),
            'default_focus_group': cfg['default_focus_group'],
        }
    )


## 2. Build the boundary and example catalogs

Boundary catalogs are built lazily per environment and cached to disk.


In [8]:
ENV_STATE_CACHE = {}

def get_env_config(env_name):
    if env_name not in ENV_CONFIGS:
        raise KeyError(f'Unknown environment: {env_name}')
    return ENV_CONFIGS[env_name]

def _preview_text(text, limit=GROUP_PREVIEW_CHARS):
    text = ' '.join(str(text).split())
    if len(text) <= limit:
        return text
    return text[:limit - 3] + '...'

def _safe_int(value):
    try:
        if value is None or (isinstance(value, float) and not np.isfinite(value)):
            return None
        return int(value)
    except Exception:
        return None

def _safe_float(value):
    try:
        if value is None:
            return None
        return float(value)
    except Exception:
        return None

def _to_bool(value):
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    return str(value).strip().lower() in {'1', 'true', 'yes', 'y'}

def normalize_localization_example_id(example_id):
    example_id = str(example_id or '')
    parts = example_id.split('/')
    if len(parts) > 3 and parts[0] == 'deception_miner':
        return '/'.join(parts[2:])
    return example_id

def compact_currency(value):
    safe_value = _safe_int(value)
    if safe_value is None:
        return 'n/a'
    return f"${safe_value:,}"

def parse_regex_group(text, pattern, *, group=1, flags=0):
    match = re.search(pattern, str(text or ''), flags=flags)
    if match is None:
        return None
    try:
        return match.group(group)
    except Exception:
        return None

def parse_shortest_directions(prompt_text):
    raw_list = parse_regex_group(
        prompt_text,
        r"Shortest direction for Explorer to destination:\s*(\[[^\n]+\])",
    )
    if raw_list is None:
        return []
    try:
        values = ast.literal_eval(raw_list)
    except Exception:
        values = re.findall(r'[A-Z]+', raw_list)
    return [str(value) for value in values]

def build_literal_summary(values):
    values = [str(value) for value in values if str(value)]
    if not values:
        return 'n/a'
    return ', '.join(values)

def load_examples_for_env(env_name):
    cfg = get_env_config(env_name)
    dataset_root = cfg['dataset_root']

    if env_name == 'bs':
        examples_df = bpc.load_bs_examples(dataset_root).copy()
        examples_df['env_name'] = env_name
        examples_df['summary_primary'] = [
            f"rank {int(rank)} | truthful {int(capacity)}"
            for rank, capacity in zip(
                examples_df['required_rank'].fillna(-1),
                examples_df['truthful_capacity'].fillna(-1),
            )
        ]
        examples_df['summary_secondary'] = [
            f"hand {bpc.cards_to_label(cards or ())}"
            for cards in examples_df['hand_cards']
        ]
        examples_df['action_summary'] = [
            f"play {bpc.cards_to_label(cards or ())}"
            for cards in examples_df['played_cards']
        ]
        return examples_df.sort_values('example_id').reset_index(drop=True)

    records = bpc.read_jsonl(dataset_root / 'examples.jsonl')
    rows = []
    for record in records:
        prompt = str(record.get('prompt') or '')
        action = record.get('action') or {}
        parsed_candidate = action.get('parsed_candidate') or {}
        action_label = str(action.get('Action') or parsed_candidate.get('Action') or '')
        is_deceptive = int(bool(_to_bool(record.get('deceptive')) or _to_bool(record.get('recorded_deceptive'))))

        row = {
            'example_id': record.get('example_id'),
            'base_example_id': record.get('base_example_id'),
            'game_id': record.get('game_id'),
            'turn_idx': record.get('turn_idx'),
            'state_id': record.get('state_id'),
            'sample_idx': record.get('sample_idx'),
            'env_name': env_name,
            'prompt': prompt,
            'prompt_text': prompt,
            'action_reasoning': record.get('action_reasoning'),
            'action_raw_text': record.get('action_raw_text'),
            'action': action,
            'action_label': action_label,
            'is_deceptive': is_deceptive,
            'source_row': record,
        }

        if env_name == 'gridworld':
            shortest_directions = parse_shortest_directions(prompt)
            recommended_direction = str(action.get('Direction') or '')
            row.update(
                {
                    'shortest_directions': shortest_directions,
                    'recommended_direction': recommended_direction,
                    'summary_primary': f"shortest {build_literal_summary(shortest_directions)}",
                    'summary_secondary': f"recommend {recommended_direction or 'n/a'}",
                    'action_summary': f"recommend {recommended_direction or 'n/a'}",
                }
            )
        elif env_name == 'interview':
            current_offer_match = parse_regex_group(prompt, r'Total comp on paper:\s*\$([0-9,]+)', flags=re.IGNORECASE)
            desired_total_match = parse_regex_group(prompt, r'DESIRED TOTAL COMP:\s*\$([0-9,]+)', flags=re.IGNORECASE)
            current_offer_total = _safe_int(current_offer_match.replace(',', '') if current_offer_match else None)
            desired_total_comp = _safe_int(desired_total_match.replace(',', '') if desired_total_match else None)
            claimed_salary_raw = action.get('Claimed_other_offer_salary')
            if claimed_salary_raw is None:
                claimed_salary_raw = parsed_candidate.get('Claimed_other_offer_salary')
            claimed_other_offer_salary = _safe_int(claimed_salary_raw)
            salary_summary = (
                f"{action_label} @ {compact_currency(claimed_other_offer_salary)}"
                if claimed_other_offer_salary is not None else (action_label or 'n/a')
            )
            row.update(
                {
                    'current_offer_total': current_offer_total,
                    'desired_total_comp': desired_total_comp,
                    'claimed_other_offer_salary': claimed_other_offer_salary,
                    'summary_primary': f"offer {compact_currency(current_offer_total)} -> target {compact_currency(desired_total_comp)}",
                    'summary_secondary': salary_summary,
                    'action_summary': salary_summary,
                }
            )
        elif env_name == 'advisor_audit':
            option_1_id = parse_regex_group(prompt, r'Option 1 \(([^)]+)\)', flags=re.IGNORECASE)
            option_2_id = parse_regex_group(prompt, r'Option 2 \(([^)]+)\)', flags=re.IGNORECASE)
            audit_probability_raw = parse_regex_group(prompt, r'Audit probability this month:\s*([0-9.]+)', flags=re.IGNORECASE)
            recommended_investment_id = (
                action.get('Recommended_investment_id')
                or parsed_candidate.get('Recommended_investment_id')
                or ''
            )
            row.update(
                {
                    'option_1_id': str(option_1_id or ''),
                    'option_2_id': str(option_2_id or ''),
                    'audit_probability': _safe_float(audit_probability_raw),
                    'recommended_investment_id': str(recommended_investment_id),
                    'summary_primary': f"audit {audit_probability_raw or 'n/a'} | {option_1_id or 'option1'} vs {option_2_id or 'option2'}",
                    'summary_secondary': f"recommend {recommended_investment_id or 'n/a'}",
                    'action_summary': f"recommend {recommended_investment_id or 'n/a'}",
                }
            )
        elif env_name == 'car_sales':
            target_issue = parse_regex_group(
                prompt,
                r'current target issue:\s*([a-zA-Z0-9_]+)',
                flags=re.IGNORECASE,
            )
            row.update(
                {
                    'target_issue': str(target_issue or ''),
                    'summary_primary': f"target issue {target_issue or 'n/a'}",
                    'summary_secondary': action_label or 'n/a',
                    'action_summary': action_label or 'n/a',
                }
            )
        else:
            row.update(
                {
                    'summary_primary': 'n/a',
                    'summary_secondary': 'n/a',
                    'action_summary': action_label or 'n/a',
                }
            )

        rows.append(row)

    examples_df = pd.DataFrame(rows).sort_values('example_id').reset_index(drop=True)
    return examples_df

def build_boundary_catalog(examples_df, *, env_name, localization_dir, cache_path, force_rebuild=False):
    cache_path = Path(cache_path)
    localization_dir = Path(localization_dir)

    if cache_path.exists() and not force_rebuild:
        cached_df = pd.read_csv(cache_path)
        if 'abs_delta' not in cached_df.columns and 'delta' in cached_df.columns:
            cached_df['abs_delta'] = cached_df['delta'].abs()
        return cached_df

    rows = []
    example_meta_lookup = examples_df.set_index('example_id')
    localization_paths = sorted(localization_dir.glob('sentence_localization_*.json'))
    total_paths = len(localization_paths)
    if total_paths == 0:
        raise RuntimeError(f'No localization files found for {env_name}: {localization_dir}')

    for file_idx, path in enumerate(localization_paths, start=1):
        if file_idx % 250 == 0:
            print(f'[{env_name} boundary catalog] processed {file_idx}/{total_paths} localization files')

        payload = json.loads(path.read_text(encoding='utf-8'))
        example_id = normalize_localization_example_id(payload.get('example_id'))
        if example_id not in example_meta_lookup.index:
            continue

        history = sorted(payload.get('history') or [], key=lambda item: int(item.get('sentence_end_idx') or 0))
        if len(history) < 2:
            continue

        example_row = example_meta_lookup.loc[example_id]
        prompt_text = str(payload.get('prompt') or example_row.get('prompt') or '')
        raw_text = str(payload.get('raw_text') or example_row.get('action_reasoning') or '')
        prev_entry = None

        for entry in history:
            rate = entry.get('deception_rate')
            prefix_idx = entry.get('sentence_end_idx')
            sentence_idx = entry.get('sentence_idx_inclusive')
            sentence_text = entry.get('sentence_text')
            if rate is None or prefix_idx is None or sentence_idx is None:
                continue

            current = {
                'example_id': str(example_id),
                'prefix_idx': int(prefix_idx),
                'sentence_idx': int(sentence_idx),
                'sentence_text': str(sentence_text or ''),
                'rate': float(rate),
            }
            if prev_entry is not None:
                delta = float(current['rate'] - prev_entry['rate'])
                rows.append(
                    {
                        'boundary_id': f"{env_name}::{example_id}::p{int(current['prefix_idx'])}",
                        'env_name': env_name,
                        'example_id': str(example_id),
                        'prefix_idx': int(current['prefix_idx']),
                        'sentence_idx': int(current['sentence_idx']),
                        'sentence_text': current['sentence_text'],
                        'rate': float(current['rate']),
                        'prev_prefix_idx': int(prev_entry['prefix_idx']),
                        'prev_sentence_idx': int(prev_entry['sentence_idx']),
                        'prev_sentence_text': prev_entry['sentence_text'],
                        'prev_rate': float(prev_entry['rate']),
                        'delta': delta,
                        'abs_delta': float(abs(delta)),
                        'prefix_gap': int(current['prefix_idx'] - prev_entry['prefix_idx']),
                        'sentence_gap': int(current['sentence_idx'] - prev_entry['sentence_idx']),
                        'game_id': example_row.get('game_id'),
                        'turn_idx': example_row.get('turn_idx'),
                        'state_id': example_row.get('state_id'),
                        'sample_idx': example_row.get('sample_idx'),
                        'summary_primary': example_row.get('summary_primary'),
                        'summary_secondary': example_row.get('summary_secondary'),
                        'action_summary': example_row.get('action_summary'),
                        'is_deceptive': int(example_row.get('is_deceptive', 0) or 0),
                        'prompt_char_len': len(prompt_text),
                        'raw_text_char_len': len(raw_text),
                        'num_localized_points': len(history),
                    }
                )
            prev_entry = current

    boundary_catalog_df = pd.DataFrame(rows)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    boundary_catalog_df.to_csv(cache_path, index=False)
    return boundary_catalog_df

def build_example_option_label(row):
    parts = [
        str(row.spike_bucket),
        f"game {row.game_id}",
        f"max Δ={float(row.max_delta):0.3f}",
        str(row.summary_primary or ''),
        str(row.summary_secondary or ''),
        _preview_text(row.top_sentence_text, 90),
    ]
    parts = [part for part in parts if part and part != 'nan']
    return ' | '.join(parts)

def build_env_state(env_name):
    cfg = get_env_config(env_name)
    examples_df = load_examples_for_env(env_name)
    example_lookup_df = examples_df.set_index('example_id')

    raw_boundary_catalog_df = build_boundary_catalog(
        examples_df,
        env_name=env_name,
        localization_dir=cfg['localization_dir'],
        cache_path=cfg['boundary_cache_path'],
        force_rebuild=FORCE_REBUILD_BOUNDARY_CACHE,
    )

    boundary_catalog_df = raw_boundary_catalog_df.copy()
    if BOUNDARY_REQUIRE_PREFIX_GAP_ONE:
        boundary_catalog_df = boundary_catalog_df.loc[boundary_catalog_df['prefix_gap'].eq(1)].copy()
    if BOUNDARY_REQUIRE_SENTENCE_GAP_ONE:
        boundary_catalog_df = boundary_catalog_df.loc[boundary_catalog_df['sentence_gap'].eq(1)].copy()

    boundary_catalog_df['abs_delta'] = boundary_catalog_df['delta'].abs()
    boundary_catalog_df = boundary_catalog_df.sort_values(['example_id', 'sentence_idx', 'prefix_idx'], ascending=[True, True, True]).reset_index(drop=True)
    if boundary_catalog_df.empty:
        raise RuntimeError(f'No boundary rows remain for {env_name} after applying the current gap filters.')

    boundary_catalog_df['boundary_option_label'] = [
        f"sent {int(row.sentence_idx):02d} | Δ={float(row.delta):+0.3f} | {_preview_text(row.sentence_text, 110)}"
        for row in boundary_catalog_df.itertuples()
    ]

    top_boundary_df = (
        boundary_catalog_df.sort_values(['example_id', 'delta', 'abs_delta', 'sentence_idx'], ascending=[True, False, False, True])
        .drop_duplicates('example_id')
        [['example_id', 'boundary_id', 'sentence_idx', 'sentence_text', 'delta']]
        .rename(columns={'boundary_id': 'top_boundary_id', 'sentence_idx': 'top_sentence_idx', 'sentence_text': 'top_sentence_text', 'delta': 'top_boundary_delta'})
    )

    flattest_boundary_df = (
        boundary_catalog_df.sort_values(['example_id', 'abs_delta', 'sentence_idx'], ascending=[True, True, True])
        .drop_duplicates('example_id')
        [['example_id', 'boundary_id', 'sentence_idx', 'sentence_text', 'delta', 'abs_delta']]
        .rename(columns={'boundary_id': 'flattest_boundary_id', 'sentence_idx': 'flattest_sentence_idx', 'sentence_text': 'flattest_sentence_text', 'delta': 'flattest_boundary_delta', 'abs_delta': 'flattest_boundary_abs_delta'})
    )

    example_summary_df = (
        boundary_catalog_df.groupby('example_id', as_index=False)
        .agg(
            max_delta=('delta', 'max'),
            min_delta=('delta', 'min'),
            max_abs_delta=('abs_delta', 'max'),
            mean_abs_delta=('abs_delta', 'mean'),
            mean_delta=('delta', 'mean'),
            num_boundaries=('boundary_id', 'count'),
            num_positive_spikes=('delta', lambda s: int((s > 0).sum())),
            num_large_spikes=('delta', lambda s: int((s >= LARGE_SPIKE_DELTA_THRESHOLD).sum())),
        )
    )

    example_meta_df = examples_df[['example_id', 'env_name', 'game_id', 'turn_idx', 'state_id', 'sample_idx', 'summary_primary', 'summary_secondary', 'action_summary', 'is_deceptive']].copy().drop_duplicates('example_id')

    example_catalog_df = example_meta_df.merge(example_summary_df, on='example_id', how='inner').merge(top_boundary_df, on='example_id', how='left').merge(flattest_boundary_df, on='example_id', how='left')

    def bucket_example(max_delta):
        if float(max_delta) <= FLAT_MAX_DELTA_THRESHOLD:
            return 'flat_or_no_spike'
        if float(max_delta) >= LARGE_SPIKE_DELTA_THRESHOLD:
            return 'large_spike'
        return 'mixed'

    example_catalog_df['spike_bucket'] = [bucket_example(value) for value in example_catalog_df['max_delta']]
    example_catalog_df['option_label'] = [build_example_option_label(row) for row in example_catalog_df.itertuples()]
    example_catalog_df = example_catalog_df.sort_values(['max_delta', 'max_abs_delta', 'example_id'], ascending=[False, False, True]).reset_index(drop=True)
    if example_catalog_df.empty:
        raise RuntimeError(f'No examples remain after building the example catalog for {env_name}.')

    return {
        'env_name': env_name,
        'examples_df': examples_df,
        'example_lookup_df': example_lookup_df,
        'boundary_catalog_df': boundary_catalog_df,
        'boundary_lookup_df': boundary_catalog_df.set_index('boundary_id'),
        'example_catalog_df': example_catalog_df,
        'example_catalog_lookup_df': example_catalog_df.set_index('example_id'),
    }

def get_env_state(env_name):
    if env_name not in ENV_STATE_CACHE:
        ENV_STATE_CACHE[env_name] = build_env_state(env_name)
    return ENV_STATE_CACHE[env_name]

def get_example_subset_df(env_name, subset_name):
    example_catalog_df = get_env_state(env_name)['example_catalog_df']
    if subset_name == 'top_spike_examples':
        return example_catalog_df.sort_values(['max_delta', 'max_abs_delta', 'example_id'], ascending=[False, False, True]).head(EXAMPLE_WIDGET_LIMIT).reset_index(drop=True)
    if subset_name == 'flat_or_no_spike_examples':
        flat_df = example_catalog_df.loc[example_catalog_df['max_delta'].le(float(FLAT_MAX_DELTA_THRESHOLD))].copy()
        if flat_df.empty:
            flat_df = example_catalog_df.nsmallest(EXAMPLE_WIDGET_LIMIT, 'max_delta').copy()
        return flat_df.sort_values(['max_delta', 'max_abs_delta', 'example_id'], ascending=[True, True, True]).reset_index(drop=True)
    return example_catalog_df.sort_values(['max_delta', 'max_abs_delta', 'example_id'], ascending=[False, False, True]).reset_index(drop=True)

def get_boundary_rows_for_example(env_name, example_id):
    boundary_catalog_df = get_env_state(env_name)['boundary_catalog_df']
    return boundary_catalog_df.loc[boundary_catalog_df['example_id'].eq(example_id)].sort_values(['sentence_idx', 'prefix_idx']).reset_index(drop=True)

def default_boundary_id_for_example(env_name, example_id, *, subset_name='top_spike_examples'):
    rows = get_boundary_rows_for_example(env_name, example_id)
    if rows.empty:
        raise KeyError(f'No boundary rows found for env={env_name}, example_id={example_id}')
    if subset_name == 'flat_or_no_spike_examples':
        return rows.sort_values(['abs_delta', 'sentence_idx', 'prefix_idx'], ascending=[True, True, True]).iloc[0]['boundary_id']
    return rows.sort_values(['delta', 'abs_delta', 'sentence_idx'], ascending=[False, False, True]).iloc[0]['boundary_id']

def build_example_boundary_overview_df(env_name, example_id, selected_boundary_id, *, max_rows=8):
    rows = get_boundary_rows_for_example(env_name, example_id).copy()
    rows['selected'] = rows['boundary_id'].eq(selected_boundary_id)
    ranked = rows.sort_values(['abs_delta', 'sentence_idx'], ascending=[False, True]).copy()
    if max_rows is not None and len(ranked) > int(max_rows):
        display_df = ranked.head(int(max_rows)).copy()
        if not bool(display_df['selected'].any()):
            selected_df = ranked.loc[ranked['selected']].head(1)
            keep_rows = max(0, int(max_rows) - len(selected_df))
            display_df = pd.concat([display_df.head(keep_rows), selected_df], ignore_index=False)
        ranked = display_df.drop_duplicates('boundary_id').sort_values(['abs_delta', 'sentence_idx'], ascending=[False, True]).head(int(max_rows)).copy()
    return ranked[['selected', 'boundary_id', 'sentence_idx', 'delta', 'prev_rate', 'rate', 'sentence_text']].reset_index(drop=True)

DEFAULT_ENV_STATE = get_env_state(DEFAULT_ENV)
DEFAULT_EXAMPLE_ID = DEFAULT_ENV_STATE['example_catalog_df'].iloc[0]['example_id']
DEFAULT_BOUNDARY_ID = default_boundary_id_for_example(DEFAULT_ENV, DEFAULT_EXAMPLE_ID)

env_availability_rows = []
for env_name in ENV_ORDER:
    cfg = get_env_config(env_name)
    env_availability_rows.append(
        {
            'env': env_name,
            'examples_present': (cfg['dataset_root'] / 'examples.jsonl').exists(),
            'localization_present': cfg['localization_dir'].exists(),
            'localization_files': len(list(cfg['localization_dir'].glob('sentence_localization_*.json'))) if cfg['localization_dir'].exists() else 0,
            'model_name': cfg['model_name'],
        }
    )

bucket_counts_df = DEFAULT_ENV_STATE['example_catalog_df'].groupby('spike_bucket', as_index=False).agg(num_examples=('example_id', 'count')).sort_values('spike_bucket').reset_index(drop=True)


## 3. Load the model

All environments use the same DeepSeek 7B model, and the notebook loads it once and reuses it across environment switches.


In [9]:
CURRENT_MODEL_BUNDLE = None

def reset_shared_model():
    global CURRENT_MODEL_BUNDLE
    CURRENT_MODEL_BUNDLE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_shared_model_bundle():
    global CURRENT_MODEL_BUNDLE
    if CURRENT_MODEL_BUNDLE is not None:
        return CURRENT_MODEL_BUNDLE

    reset_shared_model()
    CURRENT_MODEL_BUNDLE = bpc.load_attention_feature_model(
        SHARED_MODEL_NAME,
        device=MODEL_DEVICE,
        dtype_str=MODEL_DTYPE,
        load_in_4bit=False,
        trust_remote_code=MODEL_TRUST_REMOTE_CODE,
    )
    print(
        f"Loaded {CURRENT_MODEL_BUNDLE.model_name} on {CURRENT_MODEL_BUNDLE.device} "
        f"(fast tokenizer={getattr(CURRENT_MODEL_BUNDLE.tokenizer, 'is_fast', False)})"
    )
    return CURRENT_MODEL_BUNDLE

print('Shared model loader is ready. The model will load on the first analysis run.')


Shared model loader is ready. The model will load on the first analysis run.


## 4. Analysis helpers

The helpers below run one example through the selected model, align localized sentences, build sentence-offset token groups, and compute both exact `attention_features2.py` feature views and sentence-concentration / residual-similarity summaries.

In [10]:
CURRENT_EXAMPLE_KEY = None
CURRENT_EXAMPLE_BUNDLE = None

def cleanup_cached_example():
    global CURRENT_EXAMPLE_KEY, CURRENT_EXAMPLE_BUNDLE
    CURRENT_EXAMPLE_KEY = None
    CURRENT_EXAMPLE_BUNDLE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def safe_cosine(a, b):
    a = a.detach().to(torch.float32)
    b = b.detach().to(torch.float32)
    denom = a.norm() * b.norm()
    if float(denom.item()) <= 0:
        return float('nan')
    return float(torch.dot(a, b).item() / denom.item())

def entropy_stats_from_weights(weights):
    weight_array = np.asarray([float(value) for value in weights if np.isfinite(value)], dtype=float)
    if weight_array.size == 0:
        return {
            'entropy': float('nan'),
            'normalized_entropy': float('nan'),
            'max_weight': float('nan'),
            'second_weight': float('nan'),
            'top2_margin': float('nan'),
            'effective_count': float('nan'),
            'effective_fraction': float('nan'),
        }
    total_weight = float(weight_array.sum())
    if total_weight <= 0:
        return {
            'entropy': float('nan'),
            'normalized_entropy': float('nan'),
            'max_weight': float('nan'),
            'second_weight': float('nan'),
            'top2_margin': float('nan'),
            'effective_count': float('nan'),
            'effective_fraction': float('nan'),
        }
    probs = weight_array / total_weight
    positive_probs = probs[probs > 0]
    entropy_value = float(-(positive_probs * np.log(positive_probs)).sum()) if positive_probs.size else 0.0
    normalized_entropy = float(entropy_value / math.log(len(probs))) if len(probs) > 1 else 0.0
    sorted_probs = np.sort(probs)[::-1]
    max_weight = float(sorted_probs[0]) if sorted_probs.size else float('nan')
    second_weight = float(sorted_probs[1]) if sorted_probs.size > 1 else 0.0
    effective_count = float(np.exp(entropy_value))
    effective_fraction = float(effective_count / len(probs)) if len(probs) > 0 else float('nan')
    return {
        'entropy': entropy_value,
        'normalized_entropy': normalized_entropy,
        'max_weight': max_weight,
        'second_weight': second_weight,
        'top2_margin': float(max_weight - second_weight),
        'effective_count': effective_count,
        'effective_fraction': effective_fraction,
    }

def positive_shift_weights(values, *, eps=1e-6):
    value_array = np.asarray([float(value) for value in values if np.isfinite(value)], dtype=float)
    if value_array.size == 0:
        return np.asarray([], dtype=float)
    shifted = value_array - float(np.min(value_array))
    if float(shifted.sum()) <= 0:
        return np.ones_like(value_array, dtype=float)
    return shifted + float(eps)

def pretty_group_label(group_name):
    return ATTENTION_GROUP_LABELS.get(str(group_name), str(group_name).replace('_', ' '))

def localization_path_for_example_id(env_name, example_id):
    localization_dir = get_env_config(env_name)['localization_dir']
    safe_example_id = str(example_id).replace('/', '_')
    direct_candidate = localization_dir / f"sentence_localization_{safe_example_id}.json"
    if direct_candidate.exists():
        return direct_candidate
    wildcard_matches = sorted(localization_dir.glob(f"sentence_localization_*_{safe_example_id}.json"))
    if wildcard_matches:
        return wildcard_matches[0]
    raise FileNotFoundError(f'Could not find localization payload for env={env_name}, example_id={example_id} under {localization_dir}')

def load_localization_payload(env_name, example_id):
    path = localization_path_for_example_id(env_name, example_id)
    payload = json.loads(path.read_text(encoding='utf-8'))
    payload['normalized_example_id'] = normalize_localization_example_id(payload.get('example_id'))
    return payload

def extract_completion_parts(example_row, payload):
    prompt_text = str(payload.get('prompt') or example_row.get('prompt') or '')
    localized_raw_text = str(payload.get('raw_text') or example_row.get('action_reasoning') or '')
    source_row = example_row.get('source_row') or {}
    action_raw_text = str(source_row.get('action_raw_text') or example_row.get('action_raw_text') or localized_raw_text)

    reasoning_offset = -1
    if localized_raw_text:
        reasoning_offset = action_raw_text.find(localized_raw_text)

    if reasoning_offset < 0:
        full_text = prompt_text + localized_raw_text
        action_raw_text_used = localized_raw_text
        reasoning_offset = 0
    else:
        full_text = prompt_text + action_raw_text
        action_raw_text_used = action_raw_text

    reasoning_start_char = len(prompt_text) + int(reasoning_offset)
    reasoning_end_char = reasoning_start_char + len(localized_raw_text)
    return {
        'prompt_text': prompt_text,
        'localized_raw_text': localized_raw_text,
        'action_raw_text': action_raw_text_used,
        'full_text': full_text,
        'reasoning_offset': int(reasoning_offset),
        'reasoning_start_char': int(reasoning_start_char),
        'reasoning_end_char': int(reasoning_end_char),
    }

def build_sentence_df_for_example(payload, prompt_text, full_text, offsets, reasoning_offset):
    sentence_df = build_localized_sentence_df(payload)
    if sentence_df.empty:
        raise ValueError(f"{payload.get('example_id')} has no localized sentence rows.")
    sentence_df['full_start'] = sentence_df['raw_start'] + len(prompt_text) + int(reasoning_offset)
    sentence_df['full_end'] = sentence_df['raw_end'] + len(prompt_text) + int(reasoning_offset)
    sentence_df = add_span_match_columns(full_text, sentence_df)
    sentence_df = align_localized_sentences_to_tokens(offsets, sentence_df)
    return sentence_df

def get_example_analysis_bundle(env_name, example_id):
    global CURRENT_EXAMPLE_KEY, CURRENT_EXAMPLE_BUNDLE
    cache_key = (str(env_name), str(example_id))
    if CURRENT_EXAMPLE_KEY == cache_key and CURRENT_EXAMPLE_BUNDLE is not None:
        return CURRENT_EXAMPLE_BUNDLE

    cleanup_cached_example()

    env_state = get_env_state(env_name)
    example_row = env_state['example_lookup_df'].loc[example_id]
    payload = load_localization_payload(env_name, example_id)
    completion_parts = extract_completion_parts(example_row, payload)
    model_bundle = get_shared_model_bundle()
    tokenizer = model_bundle.tokenizer

    encoded = tokenizer(
        completion_parts['full_text'],
        add_special_tokens=False,
        return_tensors='pt',
        return_offsets_mapping=True,
    )
    input_ids = encoded['input_ids'].to(model_bundle.device)
    offsets = encoded['offset_mapping'][0].detach().cpu().tolist()
    token_strings = tokenizer.convert_ids_to_tokens(input_ids[0].detach().cpu().tolist())

    sentence_df = build_sentence_df_for_example(
        payload,
        completion_parts['prompt_text'],
        completion_parts['full_text'],
        offsets,
        completion_parts['reasoning_offset'],
    )
    if not sentence_df['span_matches'].all():
        bad_count = int((~sentence_df['span_matches']).sum())
        raise ValueError(f'{example_id} has {bad_count} localized sentence spans that do not match full_text.')
    if not (sentence_df['token_count'] > 0).all():
        bad_count = int((sentence_df['token_count'] == 0).sum())
        raise ValueError(f'{example_id} has {bad_count} localized sentences that failed to map to tokens.')

    with torch.no_grad():
        outputs = model_bundle.model(
            input_ids=input_ids,
            output_attentions=True,
            output_hidden_states=True,
            use_cache=False,
        )

    attentions = outputs.attentions
    hidden_states = outputs.hidden_states[1:] if getattr(outputs, 'hidden_states', None) is not None else None
    if hidden_states is None:
        raise ValueError('Model did not return hidden states.')

    CURRENT_EXAMPLE_KEY = cache_key
    CURRENT_EXAMPLE_BUNDLE = {
        'env_name': env_name,
        'payload': payload,
        'example_row': example_row,
        'prompt_text': completion_parts['prompt_text'],
        'localized_raw_text': completion_parts['localized_raw_text'],
        'action_raw_text': completion_parts['action_raw_text'],
        'full_text': completion_parts['full_text'],
        'reasoning_offset': completion_parts['reasoning_offset'],
        'reasoning_start_char': completion_parts['reasoning_start_char'],
        'reasoning_end_char': completion_parts['reasoning_end_char'],
        'input_ids': input_ids,
        'offsets': offsets,
        'token_strings': token_strings,
        'sentence_df': sentence_df,
        'attentions': attentions,
        'hidden_states': hidden_states,
        'num_layers': len(hidden_states),
        'num_heads': int(attentions[0].shape[1]),
        'model_name': model_bundle.model_name,
    }
    return CURRENT_EXAMPLE_BUNDLE

def find_block_span(text, start_marker, end_marker=None):
    start_idx = str(text).find(start_marker)
    if start_idx < 0:
        return None
    if end_marker is None:
        return (int(start_idx), int(len(text)))
    end_idx = str(text).find(end_marker, start_idx)
    if end_idx < 0:
        end_idx = len(text)
    return (int(start_idx), int(end_idx))

def find_regex_span(text, pattern, *, flags=0, group=0):
    match = re.search(pattern, str(text or ''), flags=flags)
    if match is None:
        return None
    return (int(match.start(group)), int(match.end(group)))

def find_final_action_json_span(full_text, prompt_char_len):
    suffix_text = full_text[int(prompt_char_len):]
    last_span = None
    for match in re.finditer(r'\{\s*"Action"\s*:\s*".*?"\}', suffix_text, flags=re.DOTALL):
        last_span = (int(prompt_char_len + match.start()), int(prompt_char_len + match.end()))
    if last_span is not None:
        return last_span
    start_idx = suffix_text.rfind('{"Action"')
    if start_idx < 0:
        start_idx = suffix_text.rfind('{ "Action"')
    if start_idx < 0:
        return None
    end_idx = suffix_text.find('}', start_idx)
    if end_idx < 0:
        return None
    return (int(prompt_char_len + start_idx), int(prompt_char_len + end_idx + 1))

def build_literal_span_lookup(text, values, *, search_start=0, search_end=None):
    spans = {}
    if search_end is None:
        search_end = len(text)
    cursor = int(search_start)
    for value in values:
        value_str = str(value)
        patterns = [f"'{value_str}'", f'"{value_str}"', value_str]
        value_span = None
        for pattern in patterns:
            idx = str(text).find(pattern, cursor, search_end)
            if idx < 0:
                idx = str(text).find(pattern, search_start, search_end)
            if idx >= 0:
                if pattern.startswith(("'", '"')) and pattern.endswith(("'", '"')):
                    value_span = (idx + 1, idx + 1 + len(value_str))
                else:
                    value_span = (idx, idx + len(value_str))
                cursor = value_span[1]
                break
        if value_span is not None:
            spans[value_str] = value_span
    return spans

def build_token_groups(base_bundle, boundary_row):
    full_text = base_bundle['full_text']
    offsets = base_bundle['offsets']
    sentence_df = base_bundle['sentence_df'].sort_values('sentence_idx').reset_index(drop=True)
    selected_sentence_idx = int(boundary_row['sentence_idx'])

    selected_matches = sentence_df.loc[sentence_df['sentence_idx'].eq(selected_sentence_idx)]
    if selected_matches.empty:
        raise KeyError(f'Could not find selected sentence_idx={selected_sentence_idx} in the localized sentence table.')
    selected_sentence_row = selected_matches.iloc[0]
    selected_position = int(selected_matches.index[0])
    prev_sentence_row = sentence_df.iloc[selected_position - 1] if selected_position > 0 else None

    group_rows = []

    def add_contiguous_group(group_name, span, *, group_kind, preview_text=None):
        if span is None:
            return
        start_char, end_char = int(span[0]), int(span[1])
        token_idxs = token_indices_for_char_span(offsets, start_char, end_char)
        if not token_idxs:
            return
        group_rows.append({'group_name': group_name, 'group_kind': group_kind, 'char_start': start_char, 'char_end': end_char, 'token_indices': [int(idx) for idx in token_idxs], 'token_count': len(token_idxs), 'text_preview': _preview_text(preview_text if preview_text is not None else full_text[start_char:end_char])})

    def add_token_union_group(group_name, token_indices, *, group_kind, preview_text):
        token_indices = sorted({int(idx) for idx in token_indices})
        if not token_indices:
            return
        group_rows.append({'group_name': group_name, 'group_kind': group_kind, 'char_start': np.nan, 'char_end': np.nan, 'token_indices': token_indices, 'token_count': len(token_indices), 'text_preview': _preview_text(preview_text)})

    def add_sentence_group(group_name, sentence_row, *, group_kind):
        add_contiguous_group(
            group_name,
            (int(sentence_row['full_start']), int(sentence_row['full_end'])),
            group_kind=group_kind,
            preview_text=str(sentence_row['sentence_text'] or ''),
        )

    def add_sentence_union_group(group_name, sentence_slice, *, group_kind):
        token_indices = []
        preview_parts = []
        for sentence_row in sentence_slice.itertuples():
            token_indices.extend(int(idx) for idx in sentence_row.token_indices)
            preview_parts.append(f"S{int(sentence_row.sentence_idx)}: {_preview_text(sentence_row.sentence_text, 70)}")
        add_token_union_group(group_name, token_indices, group_kind=group_kind, preview_text=' | '.join(preview_parts))

    for offset in range(1, 5):
        row_position = selected_position - offset
        if row_position < 0:
            continue
        add_sentence_group(
            f'sentence_i_minus_{offset}',
            sentence_df.iloc[row_position],
            group_kind='prior_sentence',
        )

    recent_start = max(0, selected_position - 4)
    add_sentence_union_group(
        'recent_prior_sentences',
        sentence_df.iloc[recent_start:selected_position],
        group_kind='prior_summary',
    )
    add_sentence_union_group(
        'earlier_prior_sentences',
        sentence_df.iloc[:recent_start],
        group_kind='prior_summary',
    )
    add_sentence_union_group(
        'all_prior_sentences',
        sentence_df.iloc[:selected_position],
        group_kind='prior_summary',
    )
    add_sentence_group('selected_sentence', selected_sentence_row, group_kind='selected_sentence')

    groups_df = pd.DataFrame(group_rows)
    if groups_df.empty:
        raise RuntimeError('Could not construct any token groups for the selected boundary.')
    groups_df = groups_df.drop_duplicates('group_name').reset_index(drop=True)
    return {'groups_df': groups_df, 'selected_sentence_row': selected_sentence_row, 'prev_sentence_row': prev_sentence_row}

def build_context_window_df(sentence_df, selected_sentence_idx):
    work = sentence_df.sort_values('sentence_idx').reset_index(drop=True).copy()
    work['delta_from_previous'] = work['deception_rate'].diff()
    work['role'] = 'other'
    work.loc[work['sentence_idx'].eq(int(selected_sentence_idx) - 1), 'role'] = 'previous'
    work.loc[work['sentence_idx'].eq(int(selected_sentence_idx)), 'role'] = 'selected'
    work.loc[work['sentence_idx'].eq(int(selected_sentence_idx) + 1), 'role'] = 'next'
    return work.loc[work['sentence_idx'].between(int(selected_sentence_idx) - 2, int(selected_sentence_idx) + 2)][['role', 'sentence_idx', 'deception_rate', 'delta_from_previous', 'sentence_text', 'start_token', 'end_token', 'token_count']].reset_index(drop=True)

def annotate_group_availability(groups_df, *, query_token_idx):
    groups_df = groups_df.copy()
    available_token_indices = []
    available_token_count = []
    future_token_count = []
    availability_note = []
    for token_indices in groups_df['token_indices']:
        visible = [int(idx) for idx in token_indices if int(idx) <= int(query_token_idx)]
        future_only = len(visible) == 0
        available_token_indices.append(visible)
        available_token_count.append(len(visible))
        future_token_count.append(len(token_indices) - len(visible))
        if future_only:
            availability_note.append('future-only from this query token')
        elif len(visible) < len(token_indices):
            availability_note.append('partially future from this query token')
        else:
            availability_note.append('fully available to this query token')
    groups_df['available_token_indices'] = available_token_indices
    groups_df['available_token_count'] = available_token_count
    groups_df['future_token_count'] = future_token_count
    groups_df['availability_note'] = availability_note
    return groups_df

def resolve_query_token(base_bundle, boundary_row, *, query_mode):
    group_view = build_token_groups(base_bundle, boundary_row)
    selected_sentence_row = group_view['selected_sentence_row']
    base_groups_df = group_view['groups_df'].copy()
    token_idx = int(selected_sentence_row['end_token'])
    query_label = 'last token of selected sentence'
    groups_df = annotate_group_availability(base_groups_df, query_token_idx=token_idx)
    token_str = base_bundle['token_strings'][token_idx]
    query_info_df = pd.DataFrame([
        {'field': 'env_name', 'value': base_bundle['env_name']},
        {'field': 'model_name', 'value': base_bundle['model_name']},
        {'field': 'query_mode', 'value': 'last_token_selected_sentence'},
        {'field': 'query_label', 'value': query_label},
        {'field': 'query_token_idx', 'value': token_idx},
        {'field': 'query_token_string', 'value': token_str},
        {'field': 'selected_sentence_idx', 'value': int(selected_sentence_row['sentence_idx'])},
        {'field': 'boundary_delta', 'value': float(boundary_row['delta'])},
    ])
    group_view.update({'base_groups_df': base_groups_df, 'groups_df': groups_df, 'query_token_idx': token_idx, 'query_info_df': query_info_df})
    return group_view

def build_attention_tables(attentions, groups_df, *, query_token_idx):
    attention_rows = []
    for layer_idx, layer_attn in enumerate(attentions):
        query_attn = layer_attn[0, :, query_token_idx, :].detach().to(torch.float32)
        for group_row in groups_df.itertuples():
            available_token_indices = list(group_row.available_token_indices)
            if available_token_indices:
                token_index_tensor = torch.tensor(available_token_indices, device=query_attn.device, dtype=torch.long)
                mass_by_head = query_attn.index_select(-1, token_index_tensor).sum(dim=-1)
            else:
                mass_by_head = torch.zeros(query_attn.shape[0], dtype=torch.float32, device=query_attn.device)
            for head_idx, attn_mass in enumerate(mass_by_head.detach().cpu().tolist()):
                attention_rows.append({'layer': int(layer_idx), 'head': int(head_idx), 'group_name': group_row.group_name, 'group_kind': group_row.group_kind, 'attn_mass': float(attn_mass)})
    head_group_df = pd.DataFrame(attention_rows)
    group_summary_df = head_group_df.groupby(['layer', 'group_name', 'group_kind'], as_index=False).agg(mean_attn=('attn_mass', 'mean'), max_attn=('attn_mass', 'max'), std_attn=('attn_mass', 'std')).merge(groups_df[['group_name', 'token_count', 'available_token_count', 'availability_note', 'text_preview']].drop_duplicates('group_name'), on='group_name', how='left').sort_values(['group_name', 'layer']).reset_index(drop=True)
    return head_group_df, group_summary_df

def build_attention_views(base_bundle, boundary_row, *, query_mode):
    query_view = resolve_query_token(base_bundle, boundary_row, query_mode=query_mode)
    groups_df = query_view['groups_df']
    query_token_idx = int(query_view['query_token_idx'])
    head_group_df, group_summary_df = build_attention_tables(base_bundle['attentions'], groups_df, query_token_idx=query_token_idx)

    selected_sentence_row = query_view['selected_sentence_row']
    summary_query_token_idx = int(selected_sentence_row['end_token'])
    summary_groups_df = annotate_group_availability(query_view['base_groups_df'], query_token_idx=summary_query_token_idx)
    _, sentence_group_summary_df = build_attention_tables(base_bundle['attentions'], summary_groups_df, query_token_idx=summary_query_token_idx)
    summary_query_info_df = pd.DataFrame([
        {'field': 'env_name', 'value': base_bundle['env_name']},
        {'field': 'model_name', 'value': base_bundle['model_name']},
        {'field': 'query_label', 'value': 'last token of selected sentence'},
        {'field': 'query_token_idx', 'value': summary_query_token_idx},
        {'field': 'query_token_string', 'value': base_bundle['token_strings'][summary_query_token_idx]},
        {'field': 'selected_sentence_idx', 'value': int(selected_sentence_row['sentence_idx'])},
        {'field': 'boundary_delta', 'value': float(boundary_row['delta'])},
    ])
    attention_delta_df = pd.DataFrame()
    attention_delta_sentence_indices = []
    sentence_df = base_bundle['sentence_df'].sort_values('sentence_idx').reset_index(drop=True)
    selected_sentence_matches = sentence_df.loc[sentence_df['sentence_idx'].eq(int(selected_sentence_row['sentence_idx']))]
    if not selected_sentence_matches.empty:
        selected_sentence_position = int(selected_sentence_matches.index[0])
        comparison_sentence_df = sentence_df.iloc[max(0, selected_sentence_position - 4):selected_sentence_position].copy()
        attention_delta_sentence_indices = [int(value) for value in comparison_sentence_df['sentence_idx'].tolist()]
        excluded_group_names = {'selected_sentence'}
        selected_delta_source_df = sentence_group_summary_df.loc[~sentence_group_summary_df['group_name'].isin(excluded_group_names)].copy()
        baseline_group_summary_frames = []
        for comparison_row in comparison_sentence_df.itertuples():
            comparison_query_token_idx = int(comparison_row.end_token)
            comparison_groups_df = annotate_group_availability(query_view['base_groups_df'], query_token_idx=comparison_query_token_idx)
            _, comparison_group_summary_df = build_attention_tables(base_bundle['attentions'], comparison_groups_df, query_token_idx=comparison_query_token_idx)
            comparison_group_summary_df = comparison_group_summary_df.loc[~comparison_group_summary_df['group_name'].isin(excluded_group_names)].copy()
            comparison_group_summary_df['comparison_sentence_idx'] = int(comparison_row.sentence_idx)
            baseline_group_summary_frames.append(comparison_group_summary_df)
        if baseline_group_summary_frames:
            baseline_group_summary_df = pd.concat(baseline_group_summary_frames, ignore_index=True)
            baseline_agg_df = (
                baseline_group_summary_df.groupby(['layer', 'group_name', 'group_kind'], as_index=False)
                .agg(
                    baseline_mean_attn=('mean_attn', 'mean'),
                    comparison_sentence_count=('comparison_sentence_idx', 'nunique'),
                )
            )
            attention_delta_df = (
                selected_delta_source_df[['layer', 'group_name', 'group_kind', 'mean_attn', 'available_token_count', 'availability_note', 'text_preview']]
                .rename(columns={'mean_attn': 'selected_mean_attn', 'available_token_count': 'selected_available_token_count', 'availability_note': 'selected_availability_note'})
                .merge(baseline_agg_df, on=['layer', 'group_name', 'group_kind'], how='inner')
            )
            attention_delta_df['delta_mean_attn'] = attention_delta_df['selected_mean_attn'].astype(float) - attention_delta_df['baseline_mean_attn'].astype(float)
            attention_delta_df = attention_delta_df.sort_values(['group_name', 'layer']).reset_index(drop=True)
    return {
        'groups_df': groups_df,
        'query_info_df': query_view['query_info_df'],
        'head_group_df': head_group_df,
        'group_summary_df': group_summary_df,
        'sentence_group_summary_df': sentence_group_summary_df,
        'summary_query_info_df': summary_query_info_df,
        'attention_delta_df': attention_delta_df,
        'attention_delta_sentence_indices': attention_delta_sentence_indices,
    }

def build_residual_transition_views(base_bundle, boundary_row):
    env_cfg = get_env_config(base_bundle['env_name'])
    sentence_df = base_bundle['sentence_df'].sort_values('sentence_idx').reset_index(drop=True)
    hidden_states = base_bundle['hidden_states']
    selected_sentence_idx = int(boundary_row['sentence_idx'])
    groups_df = build_token_groups(base_bundle, boundary_row)['groups_df']
    selected_sentence_matches = sentence_df.loc[sentence_df['sentence_idx'].eq(selected_sentence_idx)]
    if selected_sentence_matches.empty:
        raise RuntimeError(f'Could not locate the selected sentence row for sentence_idx={selected_sentence_idx}.')
    selected_sentence_row = selected_sentence_matches.iloc[0]
    selected_sentence_position = int(selected_sentence_matches.index[0])
    prior_sentence_df = sentence_df.iloc[:selected_sentence_position].copy()
    target_group_names = [group_name for group_name in env_cfg['residual_target_group_order'] if group_name in set(groups_df['group_name'])]
    group_token_map = {row.group_name: [int(idx) for idx in row.token_indices] for row in groups_df.itertuples() if row.group_name in target_group_names}
    layer_group_mean_cache = {}
    for layer_idx, layer_hidden in enumerate(hidden_states):
        layer_hidden_2d = layer_hidden[0].detach().to(torch.float32)
        layer_group_mean_cache[layer_idx] = {}
        for group_name in target_group_names:
            token_indices = group_token_map.get(group_name) or []
            if not token_indices:
                continue
            token_tensor = torch.tensor(token_indices, device=layer_hidden_2d.device, dtype=torch.long)
            layer_group_mean_cache[layer_idx][group_name] = layer_hidden_2d.index_select(0, token_tensor).mean(dim=0)
    transition_rows = []
    for row_idx in range(1, len(sentence_df)):
        prev_row = sentence_df.iloc[row_idx - 1]
        curr_row = sentence_df.iloc[row_idx]
        if int(curr_row['sentence_idx']) - int(prev_row['sentence_idx']) != 1:
            continue
        prev_token_idx = int(prev_row['end_token'])
        curr_token_idx = int(curr_row['end_token'])
        for layer_idx, layer_hidden in enumerate(hidden_states):
            layer_hidden_2d = layer_hidden[0].detach().to(torch.float32)
            prev_vec = layer_hidden_2d[prev_token_idx]
            curr_vec = layer_hidden_2d[curr_token_idx]
            delta_vec = curr_vec - prev_vec
            row = {'layer': int(layer_idx), 'prev_sentence_idx': int(prev_row['sentence_idx']), 'curr_sentence_idx': int(curr_row['sentence_idx']), 'transition_norm': float(delta_vec.norm().item())}
            for group_name in target_group_names:
                target_mean = layer_group_mean_cache[layer_idx].get(group_name)
                if target_mean is None:
                    continue
                direction_vec = target_mean - prev_vec
                direction_norm = float(direction_vec.norm().item())
                projection = float('nan')
                if direction_norm > 0:
                    projection = float(torch.dot(delta_vec, direction_vec / direction_vec.norm()).item())
                row[f'projection__{group_name}'] = projection
                row[f'gain__{group_name}'] = safe_cosine(curr_vec, target_mean) - safe_cosine(prev_vec, target_mean)
            transition_rows.append(row)
    transition_df = pd.DataFrame(transition_rows)
    if transition_df.empty:
        raise RuntimeError('Could not compute any consecutive sentence-boundary transition features.')
    prior_similarity_rows = []
    if not prior_sentence_df.empty:
        selected_token_idx = int(selected_sentence_row['end_token'])
        for layer_idx, layer_hidden in enumerate(hidden_states):
            layer_hidden_2d = layer_hidden[0].detach().to(torch.float32)
            selected_vec = layer_hidden_2d[selected_token_idx]
            for prior_row in prior_sentence_df.itertuples():
                prior_vec = layer_hidden_2d[int(prior_row.end_token)]
                prior_similarity_rows.append({'layer': int(layer_idx), 'selected_sentence_idx': int(selected_sentence_idx), 'prior_sentence_idx': int(prior_row.sentence_idx), 'sentence_distance': int(selected_sentence_idx - int(prior_row.sentence_idx)), 'cosine_similarity': safe_cosine(selected_vec, prior_vec), 'prior_sentence_text': str(prior_row.sentence_text or '')})
    prior_similarity_df = pd.DataFrame(prior_similarity_rows)
    metric_columns = ['transition_norm']
    metric_columns.extend([f'projection__{group_name}' for group_name in target_group_names if f'projection__{group_name}' in transition_df.columns])
    metric_columns.extend([f'gain__{group_name}' for group_name in target_group_names if f'gain__{group_name}' in transition_df.columns])
    summary_rows = []
    for layer_idx, layer_df in transition_df.groupby('layer'):
        selected_row = layer_df.loc[layer_df['curr_sentence_idx'].eq(selected_sentence_idx)].iloc[0]
        row = {'layer': int(layer_idx)}
        for column_name in metric_columns:
            if column_name not in layer_df.columns:
                continue
            row[column_name] = float(selected_row[column_name])
            std_value = float(layer_df[column_name].std(ddof=0))
            if not np.isfinite(std_value) or std_value <= 0:
                row[f'{column_name}_z'] = float('nan')
            else:
                mean_value = float(layer_df[column_name].mean())
                row[f'{column_name}_z'] = float((float(selected_row[column_name]) - mean_value) / std_value)
        summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows).sort_values('layer').reset_index(drop=True)
    if not prior_similarity_df.empty:
        prior_summary_df = prior_similarity_df.groupby('layer', as_index=False).agg(prior_sentence_cos_mean=('cosine_similarity', 'mean'), prior_sentence_cos_max=('cosine_similarity', 'max'), prior_sentence_cos_min=('cosine_similarity', 'min'))
        previous_similarity_df = prior_similarity_df.loc[prior_similarity_df['prior_sentence_idx'].eq(int(boundary_row['prev_sentence_idx']))][['layer', 'cosine_similarity']].rename(columns={'cosine_similarity': 'prior_sentence_cos_prev'})
        prior_summary_df = prior_summary_df.merge(previous_similarity_df, on='layer', how='left')
        summary_df = summary_df.merge(prior_summary_df, on='layer', how='left')
        for column_name in ['prior_sentence_cos_mean', 'prior_sentence_cos_max', 'prior_sentence_cos_min', 'prior_sentence_cos_prev']:
            if column_name not in summary_df.columns:
                continue
            std_value = float(summary_df[column_name].std(ddof=0))
            if not np.isfinite(std_value) or std_value <= 0:
                summary_df[f'{column_name}_z'] = float('nan')
            else:
                mean_value = float(summary_df[column_name].mean())
                summary_df[f'{column_name}_z'] = (summary_df[column_name].astype(float) - mean_value) / std_value
    preferred_selection_metrics = [f'gain__{group_name}_z' for group_name in env_cfg['residual_target_group_order']]
    preferred_selection_metrics.extend([f'projection__{group_name}_z' for group_name in env_cfg['residual_target_group_order']])
    preferred_selection_metrics.extend(['prior_sentence_cos_max_z', 'transition_norm_z'])
    selection_metric = None
    for metric_name in preferred_selection_metrics:
        if metric_name in summary_df.columns and summary_df[metric_name].notna().any():
            selection_metric = metric_name
            break
    if selection_metric is None:
        selection_metric = 'transition_norm_z'
    best_row_idx = int(summary_df[selection_metric].fillna(float('-inf')).idxmax())
    best_layer_number = int(summary_df.loc[best_row_idx, 'layer'])
    selected_target_group = None
    base_selection_metric = selection_metric[:-2] if selection_metric.endswith('_z') else selection_metric
    if base_selection_metric.startswith('gain__'):
        selected_target_group = base_selection_metric[len('gain__'):]
    elif base_selection_metric.startswith('projection__'):
        selected_target_group = base_selection_metric[len('projection__'):]
    best_layer_row = summary_df.loc[summary_df['layer'].eq(best_layer_number)].iloc[0]
    best_layer_target_df = pd.DataFrame([
        {'group_name': group_name, 'projection': float(best_layer_row.get(f'projection__{group_name}', np.nan)), 'projection_z': float(best_layer_row.get(f'projection__{group_name}_z', np.nan)), 'alignment_gain': float(best_layer_row.get(f'gain__{group_name}', np.nan)), 'alignment_gain_z': float(best_layer_row.get(f'gain__{group_name}_z', np.nan))}
        for group_name in target_group_names
    ])
    if not best_layer_target_df.empty:
        best_layer_target_df = best_layer_target_df.merge(groups_df[['group_name', 'group_kind', 'text_preview']].drop_duplicates('group_name'), on='group_name', how='left').sort_values('alignment_gain_z', ascending=False).reset_index(drop=True)
    prev_sentence_idx = int(boundary_row['prev_sentence_idx'])
    curr_sentence_idx = int(boundary_row['sentence_idx'])
    prev_row_lookup = sentence_df.loc[sentence_df['sentence_idx'].eq(prev_sentence_idx)]
    curr_row_lookup = sentence_df.loc[sentence_df['sentence_idx'].eq(curr_sentence_idx)]
    if prev_row_lookup.empty or curr_row_lookup.empty:
        raise RuntimeError('Could not locate the selected boundary rows inside sentence_df.')
    prev_token_idx = int(prev_row_lookup.iloc[0]['end_token'])
    curr_token_idx = int(curr_row_lookup.iloc[0]['end_token'])
    layer_hidden_2d = hidden_states[best_layer_number][0].detach().to(torch.float32)
    prev_vec = layer_hidden_2d[prev_token_idx]
    curr_vec = layer_hidden_2d[curr_token_idx]
    delta_vec = curr_vec - prev_vec
    target_direction_vec = None
    if selected_target_group is not None:
        target_mean = layer_group_mean_cache[best_layer_number].get(selected_target_group)
        if target_mean is not None:
            target_direction_vec = target_mean - prev_vec
    if target_direction_vec is None:
        target_direction_vec = torch.zeros_like(delta_vec)
    top_k = min(int(TOP_DIMENSIONS_TO_SHOW), int(delta_vec.shape[0]))
    top_dim_indices = torch.topk(delta_vec.abs(), k=top_k).indices.detach().cpu().tolist()
    top_dimension_df = pd.DataFrame([{'dimension': int(dim_idx), 'before_value': float(prev_vec[dim_idx].item()), 'after_value': float(curr_vec[dim_idx].item()), 'delta_value': float(delta_vec[dim_idx].item()), 'target_direction_value': float(target_direction_vec[dim_idx].item()), 'abs_delta_value': float(abs(delta_vec[dim_idx].item()))} for dim_idx in top_dim_indices])
    return {'summary_df': summary_df, 'prior_similarity_df': prior_similarity_df, 'target_group_names': target_group_names, 'best_layer_number': best_layer_number, 'selection_metric': selection_metric, 'selected_target_group': selected_target_group, 'best_layer_target_df': best_layer_target_df, 'top_dimension_df': top_dimension_df}

def build_focus_head_df(head_group_df, *, focus_group, focus_layer):
    focus_df = head_group_df.loc[head_group_df['group_name'].eq(focus_group) & head_group_df['layer'].eq(int(focus_layer))].copy()
    if focus_df.empty:
        return focus_df
    focus_df['head_label'] = [f'H{int(head)}' for head in focus_df['head']]
    return focus_df.sort_values('attn_mass', ascending=False).reset_index(drop=True)

def build_focus_prior_sentence_df(prior_similarity_df, *, focus_layer):
    focus_df = prior_similarity_df.loc[prior_similarity_df['layer'].eq(int(focus_layer))].copy()
    if focus_df.empty:
        return focus_df
    focus_df['sentence_label'] = [f"i-{int(row.sentence_distance)} | {_preview_text(row.prior_sentence_text, 70)}" for row in focus_df.itertuples()]
    return focus_df.sort_values(['cosine_similarity', 'sentence_distance'], ascending=[False, True]).reset_index(drop=True)

def build_sentence_attention_entropy_df(base_bundle, boundary_row):
    sentence_df = base_bundle['sentence_df'].sort_values('sentence_idx').reset_index(drop=True)
    selected_sentence_matches = sentence_df.loc[sentence_df['sentence_idx'].eq(int(boundary_row['sentence_idx']))]
    if selected_sentence_matches.empty:
        return pd.DataFrame()
    selected_sentence_position = int(selected_sentence_matches.index[0])
    context_sentence_df = sentence_df.iloc[:selected_sentence_position + 1].copy()
    query_token_idx = int(selected_sentence_matches.iloc[0]['end_token'])
    entropy_rows = []
    for layer_idx, layer_attn in enumerate(base_bundle['attentions']):
        query_attn = layer_attn[0, :, query_token_idx, :].detach().to(torch.float32).mean(dim=0)
        attention_masses = []
        sentence_indices = []
        for prior_row in context_sentence_df.itertuples():
            start_token = int(prior_row.start_token)
            end_token = int(prior_row.end_token)
            if end_token < start_token or end_token > query_token_idx:
                continue
            attention_mass = float(query_attn[start_token:end_token + 1].sum().item())
            attention_masses.append(max(0.0, attention_mass))
            sentence_indices.append(int(prior_row.sentence_idx))
        if not attention_masses:
            continue
        entropy_stats = entropy_stats_from_weights(attention_masses)
        top_sentence_idx = sentence_indices[int(np.argmax(attention_masses))]
        entropy_rows.append(
            {
                'layer': int(layer_idx),
                'num_context_sentences': int(len(sentence_indices)),
                'attention_entropy': float(entropy_stats['entropy']),
                'attention_entropy_norm': float(entropy_stats['normalized_entropy']),
                'top_sentence_attention_share': float(entropy_stats['max_weight']),
                'top_attention_sentence_margin': float(entropy_stats['top2_margin']),
                'effective_attention_sentences': float(entropy_stats['effective_count']),
                'effective_attention_fraction': float(entropy_stats['effective_fraction']),
                'top_attention_sentence_idx': int(top_sentence_idx),
                'top_attention_sentence_distance': int(int(boundary_row['sentence_idx']) - int(top_sentence_idx)),
            }
        )
    return pd.DataFrame(entropy_rows).sort_values('layer').reset_index(drop=True)

def build_prior_similarity_entropy_df(prior_similarity_df):
    if prior_similarity_df.empty:
        return pd.DataFrame()
    entropy_rows = []
    for layer_idx, layer_df in prior_similarity_df.groupby('layer'):
        ordered_df = layer_df.sort_values('prior_sentence_idx').reset_index(drop=True)
        similarity_weights = positive_shift_weights(ordered_df['cosine_similarity'].astype(float).tolist())
        if similarity_weights.size == 0:
            continue
        entropy_stats = entropy_stats_from_weights(similarity_weights)
        top_sentence_idx = int(ordered_df.iloc[int(np.argmax(similarity_weights))]['prior_sentence_idx'])
        selected_sentence_idx = int(ordered_df['selected_sentence_idx'].iloc[0])
        entropy_rows.append(
            {
                'layer': int(layer_idx),
                'num_prior_sentences': int(len(ordered_df)),
                'similarity_entropy': float(entropy_stats['entropy']),
                'similarity_entropy_norm': float(entropy_stats['normalized_entropy']),
                'top_similarity_sentence_share': float(entropy_stats['max_weight']),
                'top_similarity_sentence_margin': float(entropy_stats['top2_margin']),
                'effective_similarity_sentences': float(entropy_stats['effective_count']),
                'effective_similarity_fraction': float(entropy_stats['effective_fraction']),
                'top_similarity_sentence_idx': int(top_sentence_idx),
                'top_similarity_sentence_distance': int(selected_sentence_idx - top_sentence_idx),
                'similarity_weighting': 'min_shifted_cosine',
            }
        )
    return pd.DataFrame(entropy_rows).sort_values('layer').reset_index(drop=True)

def _feature_trace_example_id(base_bundle):
    if 'source_example_id' in base_bundle:
        return 'generated::' + '::'.join([
            str(base_bundle['source_example_id']),
            str(base_bundle.get('generation_label') or 'sample'),
            f"g{int(base_bundle.get('generation_idx', 0))}",
        ])
    payload = base_bundle.get('payload') or {}
    return str(payload.get('example_id') or base_bundle['example_row'].get('example_id') or 'feature_trace')



def _bounded_relative_change_series(current, reference):
    current_series = pd.to_numeric(current, errors='coerce').astype(float)
    reference_series = pd.to_numeric(reference, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=current_series.index, dtype=float)
    valid = current_series.notna() & reference_series.notna()
    if valid.any():
        numerator = current_series.loc[valid] - reference_series.loc[valid]
        denominator = current_series.loc[valid].abs() + reference_series.loc[valid].abs() + 1e-6
        out.loc[valid] = numerator / denominator
    return out.clip(-1.0, 1.0)


def _causal_percentile_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    for row_idx in range(len(values_series)):
        current = float(values_series.iloc[row_idx])
        if not np.isfinite(current):
            continue
        previous = values_series.iloc[:row_idx].to_numpy(dtype=float)
        previous = previous[np.isfinite(previous)]
        if previous.size == 0:
            continue
        out.iloc[row_idx] = float(np.mean(previous <= current))
    return out


def _causal_slope3_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    if len(values_series) < 3:
        return out
    xs = np.arange(3, dtype=float)
    for row_idx in range(2, len(values_series)):
        window = values_series.iloc[row_idx - 2:row_idx + 1].to_numpy(dtype=float)
        if not np.isfinite(window).all():
            continue
        slope = float(np.polyfit(xs, window, 1)[0])
        out.iloc[row_idx] = float(np.clip(slope, -1.0, 1.0))
    return out


def add_spike_feature_columns(feature_df, *, num_layers):
    df = feature_df.sort_values('sentence_idx').reset_index(drop=True).copy()
    derived_base_specs = [
        ('current_vs_prior_mean', 'g_prior_vs_self_mean'),
        ('current_vs_prev_mean', 'g_prev_vs_self_mean'),
        ('recent_vs_early_mean', 'g_early_vs_recent_mean'),
    ]
    for layer_idx in range(int(num_layers)):
        for new_stem, source_stem in derived_base_specs:
            source_column = f'{source_stem}_l{layer_idx}'
            if source_column not in df.columns:
                continue
            df[f'{new_stem}_l{layer_idx}'] = (
                1.0 - pd.to_numeric(df[source_column], errors='coerce').astype(float)
            ).clip(0.0, 1.0)
        if f'current_vs_prior_mean_l{layer_idx}' in df.columns:
            df[f'current_share_total_mean_l{layer_idx}'] = df[f'current_vs_prior_mean_l{layer_idx}']

    transition_stems = [
        'current_vs_prior_mean',
        'current_vs_prev_mean',
        'recent_vs_early_mean',
        'entropy_prior_mean',
        'top5_prior_mean',
        'herfindahl_prior_mean',
    ]
    for layer_idx in range(int(num_layers)):
        for stem in transition_stems:
            column = f'{stem}_l{layer_idx}'
            if column not in df.columns:
                continue
            current = pd.to_numeric(df[column], errors='coerce').astype(float)
            prev = current.shift(1)
            prev_running_mean = current.expanding(min_periods=1).mean().shift(1)
            prev_running_min = current.cummin().shift(1)
            prev_running_max = current.cummax().shift(1)
            df[f'delta_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev)
            df[f'devrun_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_mean)
            df[f'logratio_prev_{stem}_l{layer_idx}'] = df[f'delta_{stem}_l{layer_idx}']
            df[f'slope3_{stem}_l{layer_idx}'] = _causal_slope3_series(current)
            df[f'min_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_min)
            df[f'max_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_max)
            df[f'pct_{stem}_l{layer_idx}'] = _causal_percentile_series(current)
    return df


def _bounded_relative_change_series(current, reference):
    current_series = pd.to_numeric(current, errors='coerce').astype(float)
    reference_series = pd.to_numeric(reference, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=current_series.index, dtype=float)
    valid = current_series.notna() & reference_series.notna()
    if valid.any():
        numerator = current_series.loc[valid] - reference_series.loc[valid]
        denominator = current_series.loc[valid].abs() + reference_series.loc[valid].abs() + 1e-6
        out.loc[valid] = numerator / denominator
    return out.clip(-1.0, 1.0)


def _causal_percentile_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    for row_idx in range(len(values_series)):
        current = float(values_series.iloc[row_idx])
        if not np.isfinite(current):
            continue
        previous = values_series.iloc[:row_idx].to_numpy(dtype=float)
        previous = previous[np.isfinite(previous)]
        if previous.size == 0:
            continue
        out.iloc[row_idx] = float(np.mean(previous <= current))
    return out


def _causal_slope3_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    if len(values_series) < 3:
        return out
    xs = np.arange(3, dtype=float)
    for row_idx in range(2, len(values_series)):
        window = values_series.iloc[row_idx - 2:row_idx + 1].to_numpy(dtype=float)
        if not np.isfinite(window).all():
            continue
        slope = float(np.polyfit(xs, window, 1)[0])
        out.iloc[row_idx] = float(np.clip(slope, -1.0, 1.0))
    return out


def add_spike_feature_columns(feature_df, *, num_layers):
    df = feature_df.sort_values('sentence_idx').reset_index(drop=True).copy()
    derived_base_specs = [
        ('current_vs_prior_mean', 'g_prior_vs_self_mean'),
        ('current_vs_prev_mean', 'g_prev_vs_self_mean'),
        ('recent_vs_early_mean', 'g_early_vs_recent_mean'),
    ]
    for layer_idx in range(int(num_layers)):
        for new_stem, source_stem in derived_base_specs:
            source_column = f'{source_stem}_l{layer_idx}'
            if source_column not in df.columns:
                continue
            df[f'{new_stem}_l{layer_idx}'] = (
                1.0 - pd.to_numeric(df[source_column], errors='coerce').astype(float)
            ).clip(0.0, 1.0)
        if f'current_vs_prior_mean_l{layer_idx}' in df.columns:
            df[f'current_share_total_mean_l{layer_idx}'] = df[f'current_vs_prior_mean_l{layer_idx}']

    transition_stems = [
        'current_vs_prior_mean',
        'current_vs_prev_mean',
        'recent_vs_early_mean',
        'entropy_prior_mean',
        'top5_prior_mean',
        'herfindahl_prior_mean',
    ]
    for layer_idx in range(int(num_layers)):
        for stem in transition_stems:
            column = f'{stem}_l{layer_idx}'
            if column not in df.columns:
                continue
            current = pd.to_numeric(df[column], errors='coerce').astype(float)
            prev = current.shift(1)
            prev_running_mean = current.expanding(min_periods=1).mean().shift(1)
            prev_running_min = current.cummin().shift(1)
            prev_running_max = current.cummax().shift(1)
            df[f'delta_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev)
            df[f'devrun_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_mean)
            df[f'logratio_prev_{stem}_l{layer_idx}'] = df[f'delta_{stem}_l{layer_idx}']
            df[f'slope3_{stem}_l{layer_idx}'] = _causal_slope3_series(current)
            df[f'min_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_min)
            df[f'max_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_max)
            df[f'pct_{stem}_l{layer_idx}'] = _causal_percentile_series(current)
    return df


def _bounded_relative_change_series(current, reference):
    current_series = pd.to_numeric(current, errors='coerce').astype(float)
    reference_series = pd.to_numeric(reference, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=current_series.index, dtype=float)
    valid = current_series.notna() & reference_series.notna()
    if valid.any():
        numerator = current_series.loc[valid] - reference_series.loc[valid]
        denominator = current_series.loc[valid].abs() + reference_series.loc[valid].abs() + 1e-6
        out.loc[valid] = numerator / denominator
    return out.clip(-1.0, 1.0)


def _causal_percentile_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    for row_idx in range(len(values_series)):
        current = float(values_series.iloc[row_idx])
        if not np.isfinite(current):
            continue
        previous = values_series.iloc[:row_idx].to_numpy(dtype=float)
        previous = previous[np.isfinite(previous)]
        if previous.size == 0:
            continue
        out.iloc[row_idx] = float(np.mean(previous <= current))
    return out


def _causal_slope3_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    if len(values_series) < 3:
        return out
    xs = np.arange(3, dtype=float)
    for row_idx in range(2, len(values_series)):
        window = values_series.iloc[row_idx - 2:row_idx + 1].to_numpy(dtype=float)
        if not np.isfinite(window).all():
            continue
        slope = float(np.polyfit(xs, window, 1)[0])
        out.iloc[row_idx] = float(np.clip(slope, -1.0, 1.0))
    return out


def add_spike_feature_columns(feature_df, *, num_layers):
    df = feature_df.sort_values('sentence_idx').reset_index(drop=True).copy()
    derived_base_specs = [
        ('current_vs_prior_mean', 'g_prior_vs_self_mean'),
        ('current_vs_prev_mean', 'g_prev_vs_self_mean'),
        ('recent_vs_early_mean', 'g_early_vs_recent_mean'),
    ]
    for layer_idx in range(int(num_layers)):
        for new_stem, source_stem in derived_base_specs:
            source_column = f'{source_stem}_l{layer_idx}'
            if source_column not in df.columns:
                continue
            df[f'{new_stem}_l{layer_idx}'] = (
                1.0 - pd.to_numeric(df[source_column], errors='coerce').astype(float)
            ).clip(0.0, 1.0)
        if f'current_vs_prior_mean_l{layer_idx}' in df.columns:
            df[f'current_share_total_mean_l{layer_idx}'] = df[f'current_vs_prior_mean_l{layer_idx}']

    transition_stems = [
        'current_vs_prior_mean',
        'current_vs_prev_mean',
        'recent_vs_early_mean',
        'entropy_prior_mean',
        'top5_prior_mean',
        'herfindahl_prior_mean',
    ]
    for layer_idx in range(int(num_layers)):
        for stem in transition_stems:
            column = f'{stem}_l{layer_idx}'
            if column not in df.columns:
                continue
            current = pd.to_numeric(df[column], errors='coerce').astype(float)
            prev = current.shift(1)
            prev_running_mean = current.expanding(min_periods=1).mean().shift(1)
            prev_running_min = current.cummin().shift(1)
            prev_running_max = current.cummax().shift(1)
            df[f'delta_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev)
            df[f'devrun_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_mean)
            df[f'logratio_prev_{stem}_l{layer_idx}'] = df[f'delta_{stem}_l{layer_idx}']
            df[f'slope3_{stem}_l{layer_idx}'] = _causal_slope3_series(current)
            df[f'min_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_min)
            df[f'max_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_max)
            df[f'pct_{stem}_l{layer_idx}'] = _causal_percentile_series(current)
    return df


def _bounded_relative_change_series(current, reference):
    current_series = pd.to_numeric(current, errors='coerce').astype(float)
    reference_series = pd.to_numeric(reference, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=current_series.index, dtype=float)
    valid = current_series.notna() & reference_series.notna()
    if valid.any():
        numerator = current_series.loc[valid] - reference_series.loc[valid]
        denominator = current_series.loc[valid].abs() + reference_series.loc[valid].abs() + 1e-6
        out.loc[valid] = numerator / denominator
    return out.clip(-1.0, 1.0)


def _causal_percentile_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    for row_idx in range(len(values_series)):
        current = float(values_series.iloc[row_idx])
        if not np.isfinite(current):
            continue
        previous = values_series.iloc[:row_idx].to_numpy(dtype=float)
        previous = previous[np.isfinite(previous)]
        if previous.size == 0:
            continue
        out.iloc[row_idx] = float(np.mean(previous <= current))
    return out


def _causal_slope3_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    if len(values_series) < 3:
        return out
    xs = np.arange(3, dtype=float)
    for row_idx in range(2, len(values_series)):
        window = values_series.iloc[row_idx - 2:row_idx + 1].to_numpy(dtype=float)
        if not np.isfinite(window).all():
            continue
        slope = float(np.polyfit(xs, window, 1)[0])
        out.iloc[row_idx] = float(np.clip(slope, -1.0, 1.0))
    return out


def add_spike_feature_columns(feature_df, *, num_layers):
    df = feature_df.sort_values('sentence_idx').reset_index(drop=True).copy()
    derived_base_specs = [
        ('current_vs_prior_mean', 'g_prior_vs_self_mean'),
        ('current_vs_prev_mean', 'g_prev_vs_self_mean'),
        ('recent_vs_early_mean', 'g_early_vs_recent_mean'),
    ]
    for layer_idx in range(int(num_layers)):
        for new_stem, source_stem in derived_base_specs:
            source_column = f'{source_stem}_l{layer_idx}'
            if source_column not in df.columns:
                continue
            df[f'{new_stem}_l{layer_idx}'] = (
                1.0 - pd.to_numeric(df[source_column], errors='coerce').astype(float)
            ).clip(0.0, 1.0)
        if f'current_vs_prior_mean_l{layer_idx}' in df.columns:
            df[f'current_share_total_mean_l{layer_idx}'] = df[f'current_vs_prior_mean_l{layer_idx}']

    transition_stems = [
        'current_vs_prior_mean',
        'current_vs_prev_mean',
        'recent_vs_early_mean',
        'entropy_prior_mean',
        'top5_prior_mean',
        'herfindahl_prior_mean',
    ]
    for layer_idx in range(int(num_layers)):
        for stem in transition_stems:
            column = f'{stem}_l{layer_idx}'
            if column not in df.columns:
                continue
            current = pd.to_numeric(df[column], errors='coerce').astype(float)
            prev = current.shift(1)
            prev_running_mean = current.expanding(min_periods=1).mean().shift(1)
            prev_running_min = current.cummin().shift(1)
            prev_running_max = current.cummax().shift(1)
            df[f'delta_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev)
            df[f'devrun_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_mean)
            df[f'logratio_prev_{stem}_l{layer_idx}'] = df[f'delta_{stem}_l{layer_idx}']
            df[f'slope3_{stem}_l{layer_idx}'] = _causal_slope3_series(current)
            df[f'min_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_min)
            df[f'max_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_max)
            df[f'pct_{stem}_l{layer_idx}'] = _causal_percentile_series(current)
    return df


def _bounded_relative_change_series(current, reference):
    current_series = pd.to_numeric(current, errors='coerce').astype(float)
    reference_series = pd.to_numeric(reference, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=current_series.index, dtype=float)
    valid = current_series.notna() & reference_series.notna()
    if valid.any():
        numerator = current_series.loc[valid] - reference_series.loc[valid]
        denominator = current_series.loc[valid].abs() + reference_series.loc[valid].abs() + 1e-6
        out.loc[valid] = numerator / denominator
    return out.clip(-1.0, 1.0)


def _causal_percentile_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    for row_idx in range(len(values_series)):
        current = float(values_series.iloc[row_idx])
        if not np.isfinite(current):
            continue
        previous = values_series.iloc[:row_idx].to_numpy(dtype=float)
        previous = previous[np.isfinite(previous)]
        if previous.size == 0:
            continue
        out.iloc[row_idx] = float(np.mean(previous <= current))
    return out


def _causal_slope3_series(values):
    values_series = pd.to_numeric(values, errors='coerce').astype(float)
    out = pd.Series(np.nan, index=values_series.index, dtype=float)
    if len(values_series) < 3:
        return out
    xs = np.arange(3, dtype=float)
    for row_idx in range(2, len(values_series)):
        window = values_series.iloc[row_idx - 2:row_idx + 1].to_numpy(dtype=float)
        if not np.isfinite(window).all():
            continue
        slope = float(np.polyfit(xs, window, 1)[0])
        out.iloc[row_idx] = float(np.clip(slope, -1.0, 1.0))
    return out


def add_spike_feature_columns(feature_df, *, num_layers):
    df = feature_df.sort_values('sentence_idx').reset_index(drop=True).copy()
    derived_base_specs = [
        ('current_vs_prior_mean', 'g_prior_vs_self_mean'),
        ('current_vs_prev_mean', 'g_prev_vs_self_mean'),
        ('recent_vs_early_mean', 'g_early_vs_recent_mean'),
    ]
    for layer_idx in range(int(num_layers)):
        for new_stem, source_stem in derived_base_specs:
            source_column = f'{source_stem}_l{layer_idx}'
            if source_column not in df.columns:
                continue
            df[f'{new_stem}_l{layer_idx}'] = (
                1.0 - pd.to_numeric(df[source_column], errors='coerce').astype(float)
            ).clip(0.0, 1.0)
        if f'current_vs_prior_mean_l{layer_idx}' in df.columns:
            df[f'current_share_total_mean_l{layer_idx}'] = df[f'current_vs_prior_mean_l{layer_idx}']

    transition_stems = [
        'current_vs_prior_mean',
        'current_vs_prev_mean',
        'recent_vs_early_mean',
        'entropy_prior_mean',
        'top5_prior_mean',
        'herfindahl_prior_mean',
    ]
    for layer_idx in range(int(num_layers)):
        for stem in transition_stems:
            column = f'{stem}_l{layer_idx}'
            if column not in df.columns:
                continue
            current = pd.to_numeric(df[column], errors='coerce').astype(float)
            prev = current.shift(1)
            prev_running_mean = current.expanding(min_periods=1).mean().shift(1)
            prev_running_min = current.cummin().shift(1)
            prev_running_max = current.cummax().shift(1)
            df[f'delta_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev)
            df[f'devrun_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_mean)
            df[f'logratio_prev_{stem}_l{layer_idx}'] = df[f'delta_{stem}_l{layer_idx}']
            df[f'slope3_{stem}_l{layer_idx}'] = _causal_slope3_series(current)
            df[f'min_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_min)
            df[f'max_gap_{stem}_l{layer_idx}'] = _bounded_relative_change_series(current, prev_running_max)
            df[f'pct_{stem}_l{layer_idx}'] = _causal_percentile_series(current)
    return df
def get_feature_trace_df(base_bundle):

    cached_df = base_bundle.get('_feature_trace_df')
    if cached_df is not None:
        return cached_df

    feature_text = str(base_bundle.get('localized_raw_text') or '')
    if not feature_text.strip():
        raise ValueError('Could not build feature-aligned views because localized_raw_text is empty.')

    model_bundle = get_shared_model_bundle()
    tokenizer = model_bundle.tokenizer
    encoded = tokenizer(
        feature_text,
        add_special_tokens=False,
        return_tensors='pt',
        return_offsets_mapping=True,
    )
    input_ids = encoded['input_ids'].to(model_bundle.device)
    offsets = encoded['offset_mapping'][0].detach().cpu().tolist()

    sentence_columns = [
        column_name
        for column_name in ['sentence_idx', 'sentence_text', 'deception_rate', 'num_truthful', 'num_valid', 'raw_start', 'raw_end']
        if column_name in base_bundle['sentence_df'].columns
    ]
    sentence_df = base_bundle['sentence_df'][sentence_columns].copy()
    sentence_df['full_start'] = sentence_df['raw_start'].astype(int)
    sentence_df['full_end'] = sentence_df['raw_end'].astype(int)
    sentence_df = add_span_match_columns(feature_text, sentence_df)
    if not sentence_df['span_matches'].all():
        bad_count = int((~sentence_df['span_matches']).sum())
        raise ValueError(f'Feature-aligned sentence mapping failed because {bad_count} sentence spans no longer match the reasoning-only text.')
    sentence_df = align_localized_sentences_to_tokens(offsets, sentence_df)
    if not (sentence_df['token_count'] > 0).all():
        bad_count = int((sentence_df['token_count'] == 0).sum())
        raise ValueError(f'Feature-aligned sentence mapping failed because {bad_count} sentences could not be token-aligned.')
    sentence_df = af2.add_attention_region_columns(sentence_df, recent_window_tokens=FEATURE_RECENT_WINDOW_TOKENS)

    with torch.no_grad():
        outputs = model_bundle.model(
            input_ids=input_ids,
            output_attentions=True,
            output_hidden_states=True,
            use_cache=False,
        )

    attentions = outputs.attentions
    hidden_states = outputs.hidden_states[1:] if getattr(outputs, 'hidden_states', None) is not None else None
    feature_df = af2.compute_attention_features(
        attentions,
        hidden_states,
        sentence_df,
        example_id=_feature_trace_example_id(base_bundle),
        prompt_token_count=0,
        recent_window_tokens=FEATURE_RECENT_WINDOW_TOKENS,
    )
    feature_df = add_spike_feature_columns(feature_df, num_layers=len(hidden_states))
    base_bundle['_feature_trace_df'] = feature_df

    if 'outputs' in locals():
        del outputs
    if 'attentions' in locals():
        del attentions
    if 'hidden_states' in locals():
        del hidden_states
    del input_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return feature_df
def build_feature_metric_views(base_bundle, boundary_row):

    feature_trace_df = get_feature_trace_df(base_bundle)
    if feature_trace_df.empty:
        raise RuntimeError('Feature-aligned trace dataframe is empty for this selection.')
    selected_matches = feature_trace_df.loc[feature_trace_df['sentence_idx'].eq(int(boundary_row['sentence_idx']))]
    if selected_matches.empty:
        raise KeyError(f"Could not locate sentence_idx={int(boundary_row['sentence_idx'])} in the feature-aligned trace dataframe.")
    return {
        'base_bundle': base_bundle,
        'boundary_row': boundary_row,
        'feature_trace_df': feature_trace_df.sort_values('sentence_idx').reset_index(drop=True),
        'selected_feature_row': selected_matches.iloc[0],
        'num_layers': int(base_bundle['num_layers']),
    }
def build_layer_curve_df(selected_feature_row, column_stems, *, num_layers):
    rows = []
    for layer_idx in range(int(num_layers)):
        row = {'layer': int(layer_idx)}
        for column_stem in column_stems:
            row[column_stem] = float(selected_feature_row.get(f'{column_stem}_l{layer_idx}', np.nan))
        rows.append(row)
    return pd.DataFrame(rows)







def build_sentence_end_activation_curve_df(base_bundle, boundary_row, *, recent_window_sentences=4):
    sentence_df = base_bundle['sentence_df'].sort_values('sentence_idx').reset_index(drop=True)
    selected_sentence_idx = int(boundary_row['sentence_idx'])
    selected_matches = sentence_df.loc[sentence_df['sentence_idx'].eq(selected_sentence_idx)]
    if selected_matches.empty:
        raise KeyError(f"Could not locate sentence_idx={selected_sentence_idx} in the sentence trace.")
    selected_row = selected_matches.iloc[0]
    selected_position = int(selected_matches.index[0])
    recent_start = max(0, selected_position - int(recent_window_sentences))
    recent_rows = sentence_df.iloc[recent_start:selected_position + 1].copy().reset_index(drop=True)
    recent_sentence_indices = [int(row.sentence_idx) for row in recent_rows.itertuples()]

    def _mean_pairwise_cosine(vectors):
        if len(vectors) <= 1:
            return float('nan')
        pairwise = []
        for idx_a in range(len(vectors)):
            for idx_b in range(idx_a + 1, len(vectors)):
                pairwise.append(safe_cosine(vectors[idx_a], vectors[idx_b]))
        pairwise = [float(value) for value in pairwise if np.isfinite(float(value))]
        if not pairwise:
            return float('nan')
        return float(np.mean(pairwise))

    def _block_pca_stats(vectors):
        if len(vectors) <= 1:
            return 1.0, 0.0
        block = torch.stack(vectors, dim=0).to(dtype=torch.float32)
        centered = block - block.mean(dim=0, keepdim=True)
        singular_values = torch.linalg.svdvals(centered)
        if singular_values.numel() == 0:
            return float('nan'), float('nan')
        energy = singular_values.square()
        total_energy = float(energy.sum().item())
        if total_energy <= 1e-6:
            return 1.0, 0.0
        pc1_share = float(energy[0].item() / total_energy)
        support_k = int(energy.shape[0])
        if support_k <= 1:
            effective_rank = 0.0
        else:
            denom = float(energy.square().sum().item())
            participation_ratio = float((total_energy ** 2) / denom) if denom > 1e-6 else 1.0
            effective_rank = float(np.clip((participation_ratio - 1.0) / (support_k - 1.0), 0.0, 1.0))
        return float(np.clip(pc1_share, 0.0, 1.0)), float(effective_rank)

    rows = []
    for layer_idx, layer_hidden in enumerate(base_bundle['hidden_states']):
        hidden = layer_hidden[0].to(dtype=torch.float32)
        current_vec = hidden[int(selected_row['end_token'])]
        previous_vec = None
        if selected_position > 0:
            previous_row = sentence_df.iloc[selected_position - 1]
            previous_vec = hidden[int(previous_row['end_token'])]
        recent_vectors = [hidden[int(row.end_token)] for row in recent_rows.itertuples()]
        prior_three_vectors = recent_vectors[:-1][-3:]
        mean3_vec = torch.stack(prior_three_vectors, dim=0).mean(dim=0) if len(prior_three_vectors) >= 1 else None
        cur_norm = float(torch.linalg.vector_norm(current_vec).item())
        prev_norm = float(torch.linalg.vector_norm(previous_vec).item()) if previous_vec is not None else float('nan')
        pairwise_recent_cohesion = _mean_pairwise_cosine(recent_vectors)
        block_pc1_share, block_effective_rank = _block_pca_stats(recent_vectors)
        rows.append(
            {
                'layer': int(layer_idx),
                'selected_sentence_idx': int(selected_sentence_idx),
                'recent_sentence_indices': recent_sentence_indices,
                'cos_cur_prev': float(safe_cosine(current_vec, previous_vec)) if previous_vec is not None else float('nan'),
                'cos_cur_mean3': float(safe_cosine(current_vec, mean3_vec)) if mean3_vec is not None else float('nan'),
                'norm_cur': float(cur_norm),
                'norm_jump': float(cur_norm - prev_norm) if np.isfinite(prev_norm) else float('nan'),
                'delta_vec_norm': float(torch.linalg.vector_norm(current_vec - previous_vec).item()) if previous_vec is not None else float('nan'),
                'pairwise_recent_cohesion': float(pairwise_recent_cohesion),
                'block_pc1_share': float(block_pc1_share),
                'block_effective_rank': float(block_effective_rank),
            }
        )
    return pd.DataFrame(rows)
def plot_attention_grounding_features(feature_view):

    selected_feature_row = feature_view['selected_feature_row']
    num_layers = int(feature_view['num_layers'])
    panel_specs = [
        (
            'Current anchoring',
            [
                ('current_vs_prior_mean', 'current vs prior'),
                ('current_vs_prev_mean', 'current vs previous'),
                ('current_share_total_mean', 'current share total'),
            ],
            'Higher means the new sentence is carrying more of the relevant attention mass into itself.',
        ),
        (
            'Recency contrast',
            [
                ('recent_vs_early_mean', 'recent vs early'),
                ('prev_share_of_prior_mean', 'prev share of prior'),
            ],
            'Higher means the boundary is leaning harder on the immediately previous context than on older prior context.',
        ),
    ]
    fig, axes = plt.subplots(1, len(panel_specs), figsize=(14.8, 4.5), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, (title, metric_specs, ylabel) in zip(axes, panel_specs):
        curve_df = build_layer_curve_df(selected_feature_row, [stem for stem, _ in metric_specs], num_layers=num_layers)
        plotted_any = False
        for column_stem, label in metric_specs:
            if curve_df[column_stem].notna().any():
                line_kwargs = {'marker': 'o', 'linewidth': 1.8, 'label': label}
                if column_stem == 'current_share_total_mean':
                    line_kwargs['linestyle'] = '--'
                    line_kwargs['alpha'] = 0.7
                ax.plot(curve_df['layer'], curve_df[column_stem], **line_kwargs)
                plotted_any = True
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel('Attention share')
        ax.set_ylim(-0.02, 1.02)
        if plotted_any:
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'Metric unavailable', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
    plt.tight_layout()
    return {
        'fig': fig,
        'note': 'These are the current-vs-prior grounding features. The current-share-total trace is the same ratio as current-vs-prior, so it is drawn as a dashed sanity check rather than as a separate concept.',
    }
def plot_attention_concentration_features(feature_view):

    selected_feature_row = feature_view['selected_feature_row']
    num_layers = int(feature_view['num_layers'])
    panel_specs = [
        (
            'Entropy',
            ['entropy_full_mean', 'entropy_prior_mean'],
            ['full context', 'prior context'],
        ),
        (
            'Top-k concentration',
            ['top1_prior_mean', 'top5_prior_mean'],
            ['top1 prior', 'top5 prior'],
        ),
        (
            'Prior support concentration',
            ['herfindahl_prior_mean', 'effective_support_prior_mean'],
            ['herfindahl prior', 'effective support prior'],
        ),
    ]
    fig, axes = plt.subplots(1, len(panel_specs), figsize=(15.5, 4.6), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, (title, stems, labels) in zip(axes, panel_specs):
        curve_df = build_layer_curve_df(selected_feature_row, stems, num_layers=num_layers)
        plotted_any = False
        for column_stem, label in zip(stems, labels):
            if curve_df[column_stem].notna().any():
                ax.plot(curve_df['layer'], curve_df[column_stem], marker='o', linewidth=1.8, label=label)
                plotted_any = True
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel('Feature value')
        ax.set_ylim(-0.02, 1.02)
        if plotted_any:
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'Metric unavailable', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
    plt.tight_layout()
    return {
        'fig': fig,
        'note': 'These concentration plots focus on prior-context entropy and concentration. The prior-context top-5 share and Herfindahl score are the two most compact views of whether the model is narrowing onto a small set of earlier tokens.',
    }
def plot_activation_features(feature_view):

    base_bundle = feature_view['base_bundle']
    boundary_row = feature_view['boundary_row']
    activation_df = build_sentence_end_activation_curve_df(base_bundle, boundary_row)
    if activation_df.empty:
        return {'fig': None, 'note': 'Could not compute sentence-end activation curves for this boundary.'}

    panel_specs = [
        (
            'Sentence-end similarity to recent context',
            ['cos_cur_prev', 'cos_cur_mean3'],
            ['current vs previous', 'current vs mean of prior 3'],
            'cosine similarity',
            (-1.02, 1.02),
        ),
        (
            'Sentence-end magnitude and jump size',
            ['norm_cur', 'norm_jump', 'delta_vec_norm'],
            ['current norm', 'norm jump', 'delta vector norm'],
            'magnitude',
            None,
        ),
        (
            'Recent block structure and PCA summary',
            ['pairwise_recent_cohesion', 'block_pc1_share', 'block_effective_rank'],
            ['pairwise cohesion', 'PC1 share', 'effective rank'],
            'value',
            (-1.02, 1.02),
        ),
    ]

    fig, axes = plt.subplots(1, len(panel_specs), figsize=(18.2, 4.8), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, (title, stems, labels, ylabel, ylim) in zip(axes, panel_specs):
        plotted_any = False
        for column_stem, label in zip(stems, labels):
            if activation_df[column_stem].notna().any():
                ax.plot(activation_df['layer'], activation_df[column_stem], marker='o', linewidth=1.8, label=label)
                plotted_any = True
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel(ylabel)
        if ylim is not None:
            ax.set_ylim(*ylim)
        if plotted_any:
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'Metric unavailable', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()

    plt.tight_layout()
    return {
        'fig': fig,
        'note': 'These activation plots use the last token of the selected sentence as the anchor. The rightmost panel summarizes the raw 5-sentence dense block via PCA-style statistics, which is the block you would feed into PCA before modeling.',
    }
def plot_trace_salience_features(feature_view):

    selected_feature_row = feature_view['selected_feature_row']
    num_layers = int(feature_view['num_layers'])
    metric_specs = [
        ('current_vs_prior_mean', 'current vs prior'),
        ('current_vs_prev_mean', 'current vs previous'),
        ('recent_vs_early_mean', 'recent vs early'),
        ('entropy_prior_mean', 'prior entropy'),
        ('top5_prior_mean', 'prior top-5 share'),
        ('herfindahl_prior_mean', 'prior herfindahl'),
    ]
    delta_matrix = []
    percentile_matrix = []
    for metric_stem, _ in metric_specs:
        delta_matrix.append([float(selected_feature_row.get(f'delta_{metric_stem}_l{layer_idx}', np.nan)) for layer_idx in range(num_layers)])
        percentile_matrix.append([float(selected_feature_row.get(f'pct_{metric_stem}_l{layer_idx}', np.nan)) for layer_idx in range(num_layers)])
    delta_matrix = np.asarray(delta_matrix, dtype=float)
    percentile_matrix = np.asarray(percentile_matrix, dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(15.8, max(4.2, 0.62 * len(metric_specs))), sharex=True)
    delta_norm = matplotlib.colors.TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
    delta_image = axes[0].imshow(delta_matrix, aspect='auto', interpolation='nearest', cmap='coolwarm', norm=delta_norm)
    pct_image = axes[1].imshow(percentile_matrix, aspect='auto', interpolation='nearest', cmap='viridis', vmin=0.0, vmax=1.0)
    for ax, title in zip(axes, ['Change vs previous sentence', 'Causal percentile vs earlier sentences']):
        ax.set_xticks(np.arange(num_layers))
        tick_step = max(1, int(math.ceil(num_layers / 14)))
        tick_positions = list(range(0, num_layers, tick_step))
        if tick_positions[-1] != num_layers - 1:
            tick_positions.append(num_layers - 1)
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([str(idx) for idx in tick_positions])
        ax.set_yticks(np.arange(len(metric_specs)))
        ax.set_yticklabels([label for _, label in metric_specs])
        ax.set_xlabel('Layer')
        ax.set_title(title)
    axes[0].set_ylabel('Feature')
    delta_colorbar = fig.colorbar(delta_image, ax=axes[0], fraction=0.046, pad=0.02)
    delta_colorbar.set_label('delta feature value')
    pct_colorbar = fig.colorbar(pct_image, ax=axes[1], fraction=0.046, pad=0.02)
    pct_colorbar.set_label('causal percentile')
    plt.tight_layout()
    return {
        'fig': fig,
        'note': 'The left heatmap shows which feature families jump right at the selected boundary. The right heatmap shows whether those same values are extreme relative to the earlier trace. This view now emphasizes current-vs-prior, current-vs-previous, recent-vs-early, entropy_prior, top5_prior, and herfindahl_prior.',
    }
def figure_to_png_bytes(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=140, bbox_inches='tight')
    image_bytes = buffer.getvalue()
    plt.close(fig)
    plt.close('all')
    return image_bytes

def build_plot_section(title, what_it_shows, why_interesting, hypothesis, *, fig=None, note=None):
    return {
        'title': str(title),
        'what_it_shows': str(what_it_shows),
        'why_interesting': str(why_interesting),
        'hypothesis': str(hypothesis),
        'note': None if note is None else str(note),
        'image_png_bytes': figure_to_png_bytes(fig) if fig is not None else None,
    }

def boundary_report_header_html(report):
    return (
        "<section style='margin: 0 0 24px 0; padding: 12px 14px; border: 1px solid #ddd; background: #fafafa;'>"
        f"<div><strong>Env:</strong> {html.escape(str(report['env_name']))}</div>"
        f"<div><strong>Dataset root:</strong> {html.escape(str(report['dataset_root']))}</div>"
        f"<div><strong>Example ID:</strong> {html.escape(str(report['example_id']))}</div>"
        f"<div><strong>Boundary ID:</strong> {html.escape(str(report['boundary_id']))}</div>"
        "</section>"
    )

def describe_boundary_strength(delta_value):
    delta_value = float(delta_value)
    if delta_value >= LARGE_SPIKE_DELTA_THRESHOLD:
        return 'large positive spike'
    if abs(delta_value) <= FLAT_MAX_DELTA_THRESHOLD:
        return 'flat / near-zero boundary'
    if delta_value > 0:
        return 'moderate positive jump'
    return 'negative boundary'

def plot_localization_curve(base_bundle, boundary_row):
    sentence_df = base_bundle['sentence_df'].sort_values('sentence_idx').reset_index(drop=True)
    selected_sentence_idx = int(boundary_row['sentence_idx'])
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(sentence_df['sentence_idx'], sentence_df['deception_rate'], marker='o', linewidth=1.8, label='localized deception rate')
    selected_match = sentence_df.loc[sentence_df['sentence_idx'].eq(selected_sentence_idx)]
    if not selected_match.empty:
        ax.scatter(selected_match['sentence_idx'], selected_match['deception_rate'], color='crimson', s=90, zorder=3, label='selected sentence')
    ax.axhline(float(boundary_row['prev_rate']), color='gray', linestyle='--', linewidth=1.0)
    ax.set_title(f"Localized deception-rate trajectory (selected Δ={float(boundary_row['delta']):+0.3f})")
    ax.set_xlabel('Localized sentence index')
    ax.set_ylabel('Deception rate')
    ax.legend()
    plt.tight_layout()
    return {'fig': fig, 'note': None}

def plot_boundary_context_window(context_window_df, *, selected_sentence_idx):
    if context_window_df.empty:
        return {'fig': None, 'note': 'No local sentence context was available for this boundary.'}
    plot_df = context_window_df.sort_values('sentence_idx').reset_index(drop=True).copy()
    role_colors = {
        'other': '#9aa0a6',
        'previous': '#f4b400',
        'selected': '#db4437',
        'next': '#4285f4',
    }
    color_values = [role_colors.get(str(role), '#9aa0a6') for role in plot_df['role']]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
    axes[0].plot(plot_df['sentence_idx'], plot_df['deception_rate'], color='#34495e', marker='o', linewidth=1.8)
    axes[0].scatter(plot_df['sentence_idx'], plot_df['deception_rate'], color=color_values, s=85, zorder=3)
    axes[0].axvline(int(selected_sentence_idx), color='#db4437', linestyle='--', linewidth=1.0)
    axes[0].set_title('Local deception-rate context')
    axes[0].set_xlabel('Sentence index')
    axes[0].set_ylabel('Deception rate')
    delta_values = plot_df['delta_from_previous'].fillna(0.0).astype(float)
    axes[1].bar(plot_df['sentence_idx'], delta_values, color=color_values)
    axes[1].axhline(0.0, color='black', linestyle='--', linewidth=1.0)
    axes[1].axvline(int(selected_sentence_idx), color='#db4437', linestyle='--', linewidth=1.0)
    axes[1].set_title('Local sentence-boundary deltas')
    axes[1].set_xlabel('Sentence index')
    axes[1].set_ylabel('Δ from previous sentence')
    plt.tight_layout()
    return {'fig': fig, 'note': 'Shows the ±2 sentence neighborhood around the selected boundary, including whether the jump is isolated or part of a broader local drift.'}

def plot_example_boundary_overview(boundary_overview_df, *, selected_boundary_id):
    if boundary_overview_df.empty:
        return {'fig': None, 'note': 'No example-level boundary summary rows were available.'}
    plot_df = boundary_overview_df.copy()
    plot_df['sentence_label'] = [f"S{int(row.sentence_idx)} | {_preview_text(row.sentence_text, 72)}" for row in plot_df.itertuples()]
    plot_df['bar_color'] = [
        '#db4437' if bool(row.selected) else ('#4e79a7' if float(row.delta) >= 0 else '#9aa0a6')
        for row in plot_df.itertuples()
    ]
    plot_df = plot_df.sort_values('sentence_idx', ascending=False).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.52 * len(plot_df))))
    ax.barh(plot_df['sentence_label'], plot_df['delta'], color=plot_df['bar_color'])
    ax.axvline(0.0, color='black', linestyle='--', linewidth=1.0)
    ax.set_title('Largest boundary shifts within this example')
    ax.set_xlabel('Δ deception rate')
    plt.tight_layout()
    selected_in_plot = bool(plot_df['bar_color'].eq('#db4437').any())
    note = 'The selected boundary is highlighted in red.' if selected_in_plot else f'The selected boundary ({selected_boundary_id}) was not among the highest-magnitude example boundaries shown.'
    return {'fig': fig, 'note': note}

def plot_attention_group_summary(group_summary_df, *, env_name, summary_query_label):
    available_df = group_summary_df.loc[group_summary_df['available_token_count'] > 0].copy()
    raw_panel_specs = get_env_config(env_name)['attention_panels']
    panel_specs = []
    omitted_panel_titles = []
    for title, group_names in raw_panel_specs:
        available_group_names = [group_name for group_name in group_names if not available_df.loc[available_df['group_name'].eq(group_name)].empty]
        if available_group_names:
            panel_specs.append((title, available_group_names))
        else:
            omitted_panel_titles.append(title)
    if not panel_specs:
        note = f'No causally available attention groups were visible from the {summary_query_label}.'
        if omitted_panel_titles:
            note += f" Omitted panels: {', '.join(omitted_panel_titles)}."
        return {'fig': None, 'note': note}
    fig, axes = plt.subplots(1, len(panel_specs), figsize=(7 * len(panel_specs), 4.5), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, (title, group_names) in zip(axes, panel_specs):
        plotted_any = False
        for group_name in group_names:
            subset = available_df.loc[available_df['group_name'].eq(group_name)].sort_values('layer')
            if subset.empty:
                continue
            ax.plot(subset['layer'], subset['mean_attn'], marker='o', linewidth=1.8, label=pretty_group_label(group_name))
            plotted_any = True
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel('Mean attention mass over heads')
        if plotted_any:
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'No available groups for this panel', ha='center', va='center', transform=ax.transAxes)
    plt.tight_layout()
    note_parts = [f'Computed from the {summary_query_label}, so the full selected sentence is in context.']
    if omitted_panel_titles:
        note_parts.append(f"Omitted panels with no causally available groups: {', '.join(omitted_panel_titles)}.")
    return {'fig': fig, 'note': ' '.join(note_parts)}

def plot_attention_group_heatmap(group_summary_df, *, env_name, summary_query_label):
    available_df = group_summary_df.loc[group_summary_df['available_token_count'] > 0].copy()
    group_order = [group_name for group_name in get_env_config(env_name)['attention_group_order'] if group_name in set(available_df['group_name'])]
    if not group_order:
        return {'fig': None, 'note': f'No causally available attention groups were visible from the {summary_query_label}.'}
    heatmap_df = available_df.pivot_table(index='group_name', columns='layer', values='mean_attn', aggfunc='mean').reindex(group_order)
    if heatmap_df.empty:
        return {'fig': None, 'note': 'Could not build an attention heatmap for the selected boundary.'}
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.46 * len(heatmap_df))))
    image = ax.imshow(heatmap_df.to_numpy(dtype=float), aspect='auto', interpolation='nearest', cmap='viridis')
    layer_values = list(heatmap_df.columns)
    tick_step = max(1, int(math.ceil(len(layer_values) / 14)))
    tick_positions = list(range(0, len(layer_values), tick_step))
    if layer_values and tick_positions[-1] != len(layer_values) - 1:
        tick_positions.append(len(layer_values) - 1)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([str(layer_values[idx]) for idx in tick_positions])
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels([pretty_group_label(group_name) for group_name in heatmap_df.index])
    ax.set_xlabel('Layer')
    ax.set_ylabel('Sentence-offset group')
    ax.set_title(f'Attention heatmap by sentence-offset group ({summary_query_label})')
    colorbar = fig.colorbar(image, ax=ax, fraction=0.028, pad=0.02)
    colorbar.set_label('Mean attention mass')
    plt.tight_layout()
    return {'fig': fig, 'note': 'This condenses the sentence-offset attention summary into a single cross-layer view, which makes it easier to see whether the model concentrates on sentence i-1, pulls from a broader recent window, or spreads attention over earlier prior context.'}

def plot_attention_group_delta(attention_delta_df, *, env_name, selected_sentence_idx, comparison_sentence_indices):
    if attention_delta_df.empty:
        return {'fig': None, 'note': 'Not enough earlier sentences were available to compare the selected sentence against a prior-sentence baseline.'}
    group_order = [group_name for group_name in get_env_config(env_name)['attention_group_order'] if group_name in set(attention_delta_df['group_name'])]
    if not group_order:
        return {'fig': None, 'note': 'Could not build an attention-shift heatmap for the selected boundary.'}
    heatmap_df = attention_delta_df.pivot_table(index='group_name', columns='layer', values='delta_mean_attn', aggfunc='mean').reindex(group_order)
    if heatmap_df.empty:
        return {'fig': None, 'note': 'Could not build an attention-shift heatmap for the selected boundary.'}
    value_matrix = heatmap_df.to_numpy(dtype=float)
    finite_values = value_matrix[np.isfinite(value_matrix)]
    norm = None
    if finite_values.size:
        max_abs = float(np.nanmax(np.abs(finite_values)))
        if max_abs > 0:
            norm = matplotlib.colors.TwoSlopeNorm(vmin=-max_abs, vcenter=0.0, vmax=max_abs)
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.46 * len(heatmap_df))))
    image = ax.imshow(value_matrix, aspect='auto', interpolation='nearest', cmap='coolwarm', norm=norm)
    layer_values = list(heatmap_df.columns)
    tick_step = max(1, int(math.ceil(len(layer_values) / 14)))
    tick_positions = list(range(0, len(layer_values), tick_step))
    if layer_values and tick_positions[-1] != len(layer_values) - 1:
        tick_positions.append(len(layer_values) - 1)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([str(layer_values[idx]) for idx in tick_positions])
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels([pretty_group_label(group_name) for group_name in heatmap_df.index])
    ax.set_xlabel('Layer')
    ax.set_ylabel('Sentence-offset group')
    ax.set_title('Attention shift vs recent-sentence baseline')
    colorbar = fig.colorbar(image, ax=ax, fraction=0.028, pad=0.02)
    colorbar.set_label('Δ mean attention mass')
    plt.tight_layout()
    comparison_label = ', '.join(f'i={int(idx)}' for idx in comparison_sentence_indices) if comparison_sentence_indices else 'no earlier sentences'
    note = (
        f"Selected sentence i={int(selected_sentence_idx)} is compared against the mean of prior sentences "
        f"{comparison_label}. "
        "This keeps sentence i-1 through sentence i-4 visible, so it is easier to see whether the boundary sharpens toward a specific recent sentence or broadens out over prior context."
    )
    return {'fig': fig, 'note': note}

def plot_attention_panel_pivot(group_summary_df, *, env_name):
    available_df = group_summary_df.loc[group_summary_df['available_token_count'] > 0].copy()
    panel_specs = list(get_env_config(env_name)['attention_panels'])
    panel_rows = []
    for panel_idx, (panel_title, group_names) in enumerate(panel_specs):
        panel_group_names = [group_name for group_name in group_names if group_name in set(available_df['group_name'])]
        if not panel_group_names:
            continue
        subset = available_df.loc[available_df['group_name'].isin(panel_group_names)].copy()
        if subset.empty:
            continue
        aggregated = (
            subset.groupby('layer', as_index=False)
            .agg(
                panel_mean_attn=('mean_attn', 'mean'),
                panel_max_attn=('mean_attn', 'max'),
                contributing_groups=('group_name', 'nunique'),
            )
        )
        aggregated['panel_idx'] = int(panel_idx)
        aggregated['panel_title'] = str(panel_title)
        panel_rows.append(aggregated)
    if not panel_rows:
        return {'fig': None, 'note': 'No panel-level attention summaries were available for this boundary.'}
    panel_df = pd.concat(panel_rows, ignore_index=True).sort_values(['panel_idx', 'layer']).reset_index(drop=True)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharex=True)
    panel_titles = [title for _, title in sorted((int(row.panel_idx), str(row.panel_title)) for row in panel_df[['panel_idx', 'panel_title']].drop_duplicates().itertuples(index=False))]
    color_cycle = ['#4e79a7', '#f28e2b', '#59a14f', '#9c755f']
    panel_colors = {title: color_cycle[idx % len(color_cycle)] for idx, title in enumerate(panel_titles)}
    for panel_title in panel_titles:
        subset = panel_df.loc[panel_df['panel_title'].eq(panel_title)].sort_values('layer')
        axes[0].plot(
            subset['layer'],
            subset['panel_mean_attn'],
            marker='o',
            linewidth=1.9,
            color=panel_colors[panel_title],
            label=panel_title,
        )
    axes[0].set_title('Panel-level attention summary')
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('Mean attention mass per group')
    axes[0].legend(fontsize=8)

    contrast_specs = []
    if len(panel_specs) >= 3:
        state_title = str(panel_specs[0][0])
        reasoning_title = str(panel_specs[1][0])
        answer_title = str(panel_specs[2][0])
        available_titles = set(panel_df['panel_title'])
        if state_title in available_titles and answer_title in available_titles:
            contrast_specs.append((f'{answer_title} - {state_title}', answer_title, state_title, '#d62728'))
        if reasoning_title in available_titles and answer_title in available_titles:
            contrast_specs.append((f'{answer_title} - {reasoning_title}', answer_title, reasoning_title, '#9467bd'))
    plotted_any = False
    for label, numerator_title, denominator_title, color in contrast_specs:
        numerator_df = panel_df.loc[panel_df['panel_title'].eq(numerator_title), ['layer', 'panel_mean_attn']].rename(columns={'panel_mean_attn': 'numerator'})
        denominator_df = panel_df.loc[panel_df['panel_title'].eq(denominator_title), ['layer', 'panel_mean_attn']].rename(columns={'panel_mean_attn': 'denominator'})
        contrast_df = numerator_df.merge(denominator_df, on='layer', how='inner')
        if contrast_df.empty:
            continue
        contrast_df['contrast'] = contrast_df['numerator'].astype(float) - contrast_df['denominator'].astype(float)
        axes[1].plot(contrast_df['layer'], contrast_df['contrast'], marker='o', linewidth=1.9, color=color, label=label)
        plotted_any = True
    axes[1].axhline(0.0, color='black', linestyle='--', linewidth=1.0)
    axes[1].set_title('Answer-vs-context attention contrast')
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('Δ mean attention mass per group')
    if plotted_any:
        axes[1].legend(fontsize=8)
    else:
        axes[1].text(0.5, 0.5, 'Not enough panel coverage for contrasts', ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_axis_off()
    plt.tight_layout()
    return {'fig': fig, 'note': 'Each panel is aggregated by averaging the mean attention over the groups listed in the environment config, which makes the plot more interpretable than tracking many overlapping groups separately.'}

def plot_focus_group_heads(focus_head_df, *, focus_group, focus_layer):
    if focus_head_df.empty:
        return {'fig': None, 'note': f'No head-level attention rows for group={focus_group!r} at layer={focus_layer}.'}
    note = None
    if float(focus_head_df['attn_mass'].sum()) == 0.0:
        note = 'This focus group is future-only from the selected query token, so causal attention mass is zero by construction.'
    top_plot_df = focus_head_df.head(TOP_HEADS_TO_SHOW)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(top_plot_df['head_label'], top_plot_df['attn_mass'])
    ax.set_title(f'Head attention to {pretty_group_label(focus_group)} at layer {int(focus_layer)}')
    ax.set_xlabel('Head')
    ax.set_ylabel('Attention mass')
    plt.tight_layout()
    return {'fig': fig, 'note': note}

def plot_residual_summary(summary_df, target_group_names, selection_metric, *, env_name):
    preferred_target_groups = [group_name for group_name in get_env_config(env_name)['residual_target_group_order'] if group_name in target_group_names]
    gain_specs = [('transition_norm_z', 'transition norm z')]
    for group_name in preferred_target_groups:
        column_name = f'gain__{group_name}_z'
        if column_name in summary_df.columns:
            gain_specs.append((column_name, pretty_group_label(group_name)))
    projection_specs = []
    for group_name in preferred_target_groups:
        column_name = f'projection__{group_name}_z'
        if column_name in summary_df.columns:
            projection_specs.append((column_name, pretty_group_label(group_name)))
    prior_similarity_specs = []
    for column_name, label in [('prior_sentence_cos_max_z', 'max prior sentence cosine z'), ('prior_sentence_cos_mean_z', 'mean prior sentence cosine z'), ('prior_sentence_cos_prev_z', 'previous sentence cosine z')]:
        if column_name in summary_df.columns:
            prior_similarity_specs.append((column_name, label))
    fig, axes = plt.subplots(1, 3, figsize=(22, 4.5), sharex=True)
    for column_name, label in gain_specs:
        axes[0].plot(summary_df['layer'], summary_df[column_name], marker='o', linewidth=1.8, label=label)
    axes[0].axhline(0.0, color='black', linestyle='--', linewidth=1.0)
    axes[0].set_title('Boundary feature z-scores')
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('z-score')
    axes[0].legend(fontsize=8)
    if projection_specs:
        for column_name, label in projection_specs:
            axes[1].plot(summary_df['layer'], summary_df[column_name], marker='o', linewidth=1.8, label=label)
        axes[1].axhline(0.0, color='black', linestyle='--', linewidth=1.0)
        axes[1].set_title('Projection-to-target z-scores')
        axes[1].set_xlabel('Layer')
        axes[1].set_ylabel('z-score')
        axes[1].legend(fontsize=8)
    else:
        axes[1].text(0.5, 0.5, 'No target projection metrics available', ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('Projection-to-target z-scores')
        axes[1].set_axis_off()
    if prior_similarity_specs:
        for column_name, label in prior_similarity_specs:
            axes[2].plot(summary_df['layer'], summary_df[column_name], marker='o', linewidth=1.8, label=label)
        axes[2].axhline(0.0, color='black', linestyle='--', linewidth=1.0)
        axes[2].set_title('Prior-sentence cosine z-scores')
        axes[2].set_xlabel('Layer')
        axes[2].set_ylabel('z-score')
        axes[2].legend(fontsize=8)
    else:
        axes[2].text(0.5, 0.5, 'No prior-sentence cosine metrics available', ha='center', va='center', transform=axes[2].transAxes)
        axes[2].set_title('Prior-sentence cosine z-scores')
        axes[2].set_axis_off()
    plt.tight_layout()
    return {'fig': fig, 'note': f'Residual best-layer selection metric: {selection_metric}'}

def plot_residual_target_alignment(best_layer_target_df, best_layer_number):
    if best_layer_target_df.empty:
        return {'fig': None, 'note': 'No residual target groups were available for this example.'}
    plot_df = best_layer_target_df.copy()
    plot_df['group_label'] = [pretty_group_label(value) for value in plot_df['group_name']]
    plot_df = plot_df.sort_values('alignment_gain_z', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.45 * len(plot_df))))
    ax.barh(plot_df['group_label'], plot_df['alignment_gain_z'])
    ax.axvline(0.0, color='black', linestyle='--', linewidth=1.0)
    ax.set_title(f'Best-layer target alignment z-scores (layer {int(best_layer_number)})')
    ax.set_xlabel('alignment_gain_z')
    plt.tight_layout()
    return {'fig': fig, 'note': None}

def plot_residual_target_competition(summary_df, target_group_names, *, env_name):
    group_order = [group_name for group_name in get_env_config(env_name)['residual_target_group_order'] if group_name in target_group_names]
    gain_columns = [(group_name, f'gain__{group_name}_z') for group_name in group_order if f'gain__{group_name}_z' in summary_df.columns]
    projection_columns = [(group_name, f'projection__{group_name}_z') for group_name in group_order if f'projection__{group_name}_z' in summary_df.columns]
    metric_type = None
    metric_columns = gain_columns
    if metric_columns:
        metric_type = 'alignment_gain_z'
    elif projection_columns:
        metric_columns = projection_columns
        metric_type = 'projection_z'
    if not metric_columns:
        return {'fig': None, 'note': 'No target-competition metrics were available across layers for this boundary.'}
    competition_rows = []
    for row in summary_df.itertuples():
        score_pairs = []
        for group_name, column_name in metric_columns:
            score_value = getattr(row, column_name, np.nan)
            if np.isfinite(score_value):
                score_pairs.append((str(group_name), float(score_value)))
        if not score_pairs:
            continue
        score_pairs.sort(key=lambda item: item[1], reverse=True)
        winner_group, winner_score = score_pairs[0]
        runner_up_score = score_pairs[1][1] if len(score_pairs) > 1 else float('nan')
        competition_rows.append(
            {
                'layer': int(row.layer),
                'winner_group': winner_group,
                'winner_score': float(winner_score),
                'runner_up_score': float(runner_up_score),
                'winner_margin': float(winner_score - runner_up_score) if np.isfinite(runner_up_score) else float('nan'),
            }
        )
    competition_df = pd.DataFrame(competition_rows).sort_values('layer').reset_index(drop=True)
    if competition_df.empty:
        return {'fig': None, 'note': 'No finite target-competition rows were available for this boundary.'}
    competition_df['winner_group_label'] = [pretty_group_label(group_name) for group_name in competition_df['winner_group']]
    y_labels = [pretty_group_label(group_name) for group_name, _ in metric_columns]
    y_lookup = {label: idx for idx, label in enumerate(y_labels)}
    competition_df['winner_y'] = [y_lookup[label] for label in competition_df['winner_group_label']]
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharex=True, gridspec_kw={'width_ratios': [1.1, 1.2]})
    axes[0].plot(competition_df['layer'], competition_df['winner_score'], marker='o', linewidth=1.8, color='#d62728', label='top target')
    axes[0].plot(competition_df['layer'], competition_df['runner_up_score'], marker='o', linewidth=1.6, color='#4e79a7', label='runner-up')
    if competition_df['winner_margin'].notna().any():
        axes[0].plot(competition_df['layer'], competition_df['winner_margin'], marker='o', linewidth=1.6, color='#59a14f', label='margin')
    axes[0].axhline(0.0, color='black', linestyle='--', linewidth=1.0)
    axes[0].set_title('Target competition across layers')
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel(metric_type)
    axes[0].legend(fontsize=8)

    margin_values = competition_df['winner_margin'].to_numpy(dtype=float)
    if np.isfinite(margin_values).any():
        scatter = axes[1].scatter(
            competition_df['layer'],
            competition_df['winner_y'],
            c=competition_df['winner_margin'],
            cmap='viridis',
            s=85,
            edgecolors='black',
            linewidths=0.4,
        )
        colorbar = fig.colorbar(scatter, ax=axes[1], fraction=0.035, pad=0.02)
        colorbar.set_label('winner margin')
    else:
        axes[1].scatter(
            competition_df['layer'],
            competition_df['winner_y'],
            color='#4e79a7',
            s=85,
            edgecolors='black',
            linewidths=0.4,
        )
    axes[1].set_yticks(np.arange(len(y_labels)))
    axes[1].set_yticklabels(y_labels)
    axes[1].set_title('Winning target group by layer')
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('Winning target')
    plt.tight_layout()
    return {'fig': fig, 'note': f'Competition is measured using {metric_type}. A stable winner with a growing margin is usually easier to interpret than several similarly sized curves.'}

def plot_residual_top_dimensions(top_dimension_df, *, best_layer_number, selection_metric, selected_target_group):
    if top_dimension_df.empty:
        return {'fig': None, 'note': 'No top-dimension residual rows were available for this boundary.'}
    plot_df = top_dimension_df.sort_values(['abs_delta_value', 'dimension'], ascending=[False, True]).reset_index(drop=True).copy()
    plot_df['dimension_label'] = [f'd{int(row.dimension)}' for row in plot_df.itertuples()]
    y_positions = np.arange(len(plot_df))
    move_colors = ['#db4437' if float(value) >= 0 else '#4285f4' for value in plot_df['delta_value']]
    target_colors = ['#59a14f' if float(value) >= 0 else '#9c755f' for value in plot_df['target_direction_value']]
    fig, axes = plt.subplots(1, 2, figsize=(15, max(4.2, 0.38 * len(plot_df))), sharey=True, gridspec_kw={'width_ratios': [1.45, 1.0]})
    for y_pos, row, color in zip(y_positions, plot_df.itertuples(), move_colors):
        axes[0].plot([row.before_value, row.after_value], [y_pos, y_pos], color=color, linewidth=2.0, alpha=0.9)
        axes[0].scatter(row.before_value, y_pos, color='#9aa0a6', s=34, zorder=3)
        axes[0].scatter(row.after_value, y_pos, color=color, s=48, zorder=3)
    axes[0].axvline(0.0, color='black', linestyle='--', linewidth=1.0)
    axes[0].set_yticks(y_positions)
    axes[0].set_yticklabels(plot_df['dimension_label'])
    axes[0].invert_yaxis()
    axes[0].set_xlabel('Residual value')
    axes[0].set_title(f'Largest-moving dimensions (layer {int(best_layer_number)})')
    axes[1].barh(y_positions, plot_df['target_direction_value'], color=target_colors)
    axes[1].axvline(0.0, color='black', linestyle='--', linewidth=1.0)
    axes[1].set_xlabel('Target-direction value')
    axes[1].set_title('Same dimensions in target direction')
    axes[1].set_yticks(y_positions)
    axes[1].set_yticklabels(plot_df['dimension_label'])
    plt.tight_layout()
    note_parts = [f'Dimensions are ranked by absolute residual movement at best layer {int(best_layer_number)} (selected by {selection_metric}).']
    if selected_target_group is not None:
        note_parts.append(f'The right panel compares those same dimensions to the target-direction vector for {pretty_group_label(selected_target_group)}.')
    else:
        note_parts.append('No target group won the best-layer heuristic, so the comparison panel uses a zero target-direction baseline.')
    return {'fig': fig, 'note': ' '.join(note_parts)}

def plot_prior_sentence_similarity_heatmap(prior_similarity_df):
    if prior_similarity_df.empty:
        return {'fig': None, 'note': 'No prior-sentence cosine rows were available for a cross-layer heatmap.'}
    heatmap_df = prior_similarity_df.pivot_table(index='sentence_distance', columns='layer', values='cosine_similarity', aggfunc='mean').sort_index()
    if heatmap_df.empty:
        return {'fig': None, 'note': 'Could not build a prior-sentence similarity heatmap for this boundary.'}
    value_matrix = heatmap_df.to_numpy(dtype=float)
    finite_values = value_matrix[np.isfinite(value_matrix)]
    norm = None
    if finite_values.size:
        min_value = float(np.nanmin(finite_values))
        max_value = float(np.nanmax(finite_values))
        if min_value < 0.0 < max_value:
            norm = matplotlib.colors.TwoSlopeNorm(vmin=min_value, vcenter=0.0, vmax=max_value)
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.38 * len(heatmap_df))))
    image = ax.imshow(value_matrix, aspect='auto', interpolation='nearest', cmap='coolwarm', norm=norm)
    layer_values = list(heatmap_df.columns)
    tick_step = max(1, int(math.ceil(len(layer_values) / 14)))
    tick_positions = list(range(0, len(layer_values), tick_step))
    if layer_values and tick_positions[-1] != len(layer_values) - 1:
        tick_positions.append(len(layer_values) - 1)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([str(layer_values[idx]) for idx in tick_positions])
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels([f"i-{int(idx)}" for idx in heatmap_df.index])
    ax.set_xlabel('Layer')
    ax.set_ylabel('Prior sentence distance')
    ax.set_title('Prior-sentence similarity heatmap across layers')
    colorbar = fig.colorbar(image, ax=ax, fraction=0.028, pad=0.02)
    colorbar.set_label('cosine similarity')
    plt.tight_layout()
    return {'fig': fig, 'note': 'This shows whether the selected sentence starts to look like a specific earlier sentence only in a narrow band of layers, or whether that reactivation is broad and persistent.'}

def plot_sentence_entropy_summary(attention_entropy_df, prior_similarity_entropy_df):
    if attention_entropy_df.empty and prior_similarity_entropy_df.empty:
        return {'fig': None, 'note': 'No sentence concentration summaries were available for this boundary.'}
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    plot_specs = [
        ('attention_entropy_norm', 'similarity_entropy_norm', 'Normalized entropy', 'Lower means more concentrated'),
        ('top_sentence_attention_share', 'top_similarity_sentence_share', 'Top prior-sentence share', 'Higher means one sentence dominates more strongly'),
        ('effective_attention_sentences', 'effective_similarity_sentences', 'Effective prior-sentence count', 'Interpretable concentration as exp(entropy)'),
        ('top_attention_sentence_distance', 'top_similarity_sentence_distance', 'Distance to winning sentence', '0 means sentence i; 1 means sentence i-1'),
    ]
    for ax, (attention_col, similarity_col, title, ylabel) in zip(axes.flatten(), plot_specs):
        plotted_any = False
        if not attention_entropy_df.empty and attention_col in attention_entropy_df.columns:
            ax.plot(attention_entropy_df['layer'], attention_entropy_df[attention_col], marker='o', linewidth=1.8, color='#1f77b4', label='attention incl. self')
            plotted_any = True
        if not prior_similarity_entropy_df.empty and similarity_col in prior_similarity_entropy_df.columns:
            ax.plot(prior_similarity_entropy_df['layer'], prior_similarity_entropy_df[similarity_col], marker='o', linewidth=1.8, color='#d62728', label='similarity')
            plotted_any = True
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel(ylabel)
        if attention_col.endswith('_norm') or similarity_col.endswith('_norm'):
            ax.set_ylim(-0.02, 1.02)
        if plotted_any:
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'Metric unavailable', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
    plt.tight_layout()
    return {'fig': fig, 'note': 'The attention curves now include sentence i itself, so 0 on the distance plot means the selected sentence won. The similarity curves still derive per-layer weights by subtracting the minimum cosine value within each heatmap column before computing concentration statistics, which keeps the residual-side summary focused on contrast rather than raw scale.'}

def plot_prior_sentence_winner_trace(attention_entropy_df, prior_similarity_entropy_df):
    if attention_entropy_df.empty and prior_similarity_entropy_df.empty:
        return {'fig': None, 'note': 'No sentence winner traces were available for this boundary.'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
    winner_specs = [
        ('top_attention_sentence_distance', 'top_similarity_sentence_distance', 'Distance to winning sentence', '0 means sentence i; 1 means sentence i-1'),
        ('top_attention_sentence_margin', 'top_similarity_sentence_margin', 'Winner margin over runner-up', 'Higher means the winning prior sentence is more distinct'),
    ]
    for ax, (attention_col, similarity_col, title, ylabel) in zip(axes, winner_specs):
        plotted_any = False
        if not attention_entropy_df.empty and attention_col in attention_entropy_df.columns:
            ax.plot(attention_entropy_df['layer'], attention_entropy_df[attention_col], marker='o', linewidth=1.8, color='#1f77b4', label='attention incl. self')
            plotted_any = True
        if not prior_similarity_entropy_df.empty and similarity_col in prior_similarity_entropy_df.columns:
            ax.plot(prior_similarity_entropy_df['layer'], prior_similarity_entropy_df[similarity_col], marker='o', linewidth=1.8, color='#d62728', label='similarity')
            plotted_any = True
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel(ylabel)
        if plotted_any:
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'Metric unavailable', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
    plt.tight_layout()
    return {'fig': fig, 'note': 'The attention side now includes sentence i itself, so a distance of 0 means the selected sentence won the layer. That makes it easy to tell whether the model is mostly self-reinforcing or actually pulling from an earlier sentence.'}

def plot_prior_sentence_similarity(focus_prior_sentence_df, *, focus_layer):
    if focus_prior_sentence_df.empty:
        return {'fig': None, 'note': f'No prior-sentence cosine rows are available at layer {focus_layer}.'}
    ordered_df = focus_prior_sentence_df.sort_values('sentence_distance').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(ordered_df['sentence_distance'], ordered_df['cosine_similarity'], marker='o', linewidth=1.8)
    ax.axhline(0.0, color='black', linestyle='--', linewidth=1.0)
    ax.set_title(f'Cosine similarity to prior sentence residuals at layer {int(focus_layer)}')
    ax.set_xlabel('Prior sentence distance')
    ax.set_xticks(ordered_df['sentence_distance'])
    ax.set_xticklabels([f"i-{int(value)}" for value in ordered_df['sentence_distance']])
    ax.set_ylabel('cosine similarity')
    plt.tight_layout()
    return {'fig': fig, 'note': None}

def build_boundary_analysis_report(env_name, boundary_id, *, query_mode, focus_group, focus_layer):
    env_state = get_env_state(env_name)
    env_cfg = get_env_config(env_name)
    boundary_row = env_state['boundary_lookup_df'].loc[boundary_id]
    base_bundle = get_example_analysis_bundle(env_name, boundary_row['example_id'])
    effective_focus_layer = max(0, min(int(focus_layer), int(base_bundle['num_layers']) - 1))
    attention_view = build_attention_views(base_bundle, boundary_row, query_mode=query_mode)
    feature_view = build_feature_metric_views(base_bundle, boundary_row)
    residual_view = build_residual_transition_views(base_bundle, boundary_row)
    focus_head_df = build_focus_head_df(attention_view['head_group_df'], focus_group=focus_group, focus_layer=effective_focus_layer)
    focus_prior_sentence_df = build_focus_prior_sentence_df(residual_view['prior_similarity_df'], focus_layer=effective_focus_layer)
    attention_entropy_df = build_sentence_attention_entropy_df(base_bundle, boundary_row)
    prior_similarity_entropy_df = build_prior_similarity_entropy_df(residual_view['prior_similarity_df'])
    context_window_df = build_context_window_df(base_bundle['sentence_df'], int(boundary_row['sentence_idx']))
    boundary_overview_df = build_example_boundary_overview_df(env_name, boundary_row['example_id'], boundary_id)
    sections = []

    localization_plot = plot_localization_curve(base_bundle, boundary_row)
    sections.append(
        build_plot_section(
            'Localized deception-rate trajectory',
            'The localized deception rate sentence by sentence for this example, with the selected sentence highlighted and the previous sentence rate shown as a dashed reference.',
            'This tells you exactly where the reasoning starts to look more deceptive, so you can anchor the rest of the analysis on the boundary you care about.',
            'If deception turns on at a particular sentence boundary, we should see a discrete jump or inflection rather than a completely smooth drift across the whole reasoning trace.',
            fig=localization_plot['fig'],
            note=localization_plot['note'],
        )
    )

    context_window_plot = plot_boundary_context_window(context_window_df, selected_sentence_idx=int(boundary_row['sentence_idx']))
    sections.append(
        build_plot_section(
            'Local boundary context',
            'A zoomed-in view of the selected sentence and its immediate neighbors, showing both the deception-rate trajectory and the per-boundary jump size.',
            'This helps distinguish a sharp onset of deception from a slower drift that started one or two sentences earlier.',
            'If this is the real onset, the selected boundary should usually look like a local delta peak instead of just one point along a gradual slope.',
            fig=context_window_plot['fig'],
            note=context_window_plot['note'],
        )
    )

    boundary_overview_plot = plot_example_boundary_overview(boundary_overview_df, selected_boundary_id=boundary_id)
    sections.append(
        build_plot_section(
            'Largest shifts within the example',
            'The highest-magnitude sentence boundaries for the current example, with the selected boundary highlighted when it is among them.',
            'This helps answer whether the chosen boundary is the main deception event in the example or just one of several competing spikes.',
            'If the chosen sentence is causally important, its boundary should often rank near the largest within-example shifts rather than looking ordinary in context.',
            fig=boundary_overview_plot['fig'],
            note=boundary_overview_plot['note'],
        )
    )

    attention_plot = plot_attention_group_summary(
        attention_view['sentence_group_summary_df'],
        env_name=env_name,
        summary_query_label=str(attention_view['summary_query_info_df'].loc[attention_view['summary_query_info_df']['field'].eq('query_label'), 'value'].iloc[0]),
    )
    sections.append(
        build_plot_section(
            'Sentence-offset attention summary',
            'For each layer, the mean amount of attention the last token of the selected sentence sends to sentence i-1 through sentence i-4 plus broader prior-context buckets such as recent prior sentences and all prior sentences.',
            'This is the robust, environment-agnostic version of the old token-group view. It makes it easier to see whether the model is leaning on the immediately previous sentence, a short recent window, or a broader earlier context.',
            'If the selected sentence is genuinely grounded in a nearby prior thought, one or two recent sentence-offset curves should stand out in a limited layer band rather than every prior-context curve moving together.',
            fig=attention_plot['fig'],
            note=attention_plot['note'],
        )
    )

    attention_heatmap_plot = plot_attention_group_heatmap(
        attention_view['sentence_group_summary_df'],
        env_name=env_name,
        summary_query_label=str(attention_view['summary_query_info_df'].loc[attention_view['summary_query_info_df']['field'].eq('query_label'), 'value'].iloc[0]),
    )
    sections.append(
        build_plot_section(
            'Sentence-offset attention heatmap',
            'A compact layer-by-group heatmap of the same summary, now expressed only in terms of sentence i-1 through sentence i-4 and broader prior-context groups.',
            'This compresses the layerwise picture into one view, which makes it easier to spot when the model sharply locks onto a particular earlier sentence instead of using prior context diffusely.',
            'If the boundary reflects a focused reuse of earlier reasoning, the heatmap should show a clear hot band for a small number of sentence-offset groups rather than a uniformly warm block everywhere.',
            fig=attention_heatmap_plot['fig'],
            note=attention_heatmap_plot['note'],
        )
    )

    attention_delta_plot = plot_attention_group_delta(
        attention_view['attention_delta_df'],
        env_name=env_name,
        selected_sentence_idx=int(boundary_row['sentence_idx']),
        comparison_sentence_indices=attention_view['attention_delta_sentence_indices'],
    )
    sections.append(
        build_plot_section(
            'Attention shift vs recent-sentence baseline',
            'A layer-by-group heatmap of how the selected sentence changes relative to the mean of the prior one to four sentences, while keeping sentence i-1 through sentence i-4 visible.',
            'This is the cleanest way to ask whether the selected boundary makes a specific prior sentence more salient than it had been in the nearby reasoning history.',
            'If this boundary marks a real pivot, one or two sentence-offset groups should rise above the recent baseline right here instead of staying high throughout the whole trace.',
            fig=attention_delta_plot['fig'],
            note=attention_delta_plot['note'],
        )
    )

    focus_head_plot = plot_focus_group_heads(focus_head_df, focus_group=focus_group, focus_layer=effective_focus_layer)
    sections.append(
        build_plot_section(
            'Head attention to sentence i',
            'The strongest individual attention heads at the chosen layer for sentence i, which is the selected sentence itself.',
            'This is useful when the layer-average curves suggest self-attention is doing something special and you want to see whether that preference is carried by a small number of standout heads.',
            'If the effect is circuit-like rather than purely diffuse, a few heads should stand out clearly over the rest for sentence i.',
            fig=focus_head_plot['fig'],
            note=focus_head_plot['note'],
        )
    )

    attention_grounding_plot = plot_attention_grounding_features(feature_view)
    sections.append(
        build_plot_section(
            'Attention grounding features',
            'The current-vs-prior grounding features, including current-vs-prior, current-vs-previous, recent-vs-early, and the current-share-total sanity check.',
            'These curves are the direct feature motivation for replacing task-specific prompt groups with sentence-relative groups: they describe the same intuition in a way that transfers across environments.',
            'If the selected sentence is grounded in very recent reasoning, the sentence-i-1 and recent-prior curves should rise together while earlier-vs-recent falls in the informative layer band.',
            fig=attention_grounding_plot['fig'],
            note=attention_grounding_plot['note'],
        )
    )

    attention_concentration_plot = plot_attention_concentration_features(feature_view)
    sections.append(
        build_plot_section(
            'Attention concentration features',
            'The exact concentration-style attention features from attention_features2.py, showing entropy, top-k mass, and effective support for full context, prior context, and the selected sentence itself.',
            'These are the right companions to the grounding ratios because they show whether the model is using prior context diffusely or concentrating on a small set of tokens when deception starts to rise.',
            'If the model is locking onto a narrow prior rationale, entropy and effective support should fall while top-k mass rises in the same layer range.',
            fig=attention_concentration_plot['fig'],
            note=attention_concentration_plot['note'],
        )
    )

    activation_plot = plot_activation_features(feature_view)
    sections.append(
        build_plot_section(
            'Activation features',
            'The sentence-end activation features computed from the last token of the selected sentence, including cosine to the previous sentence, cosine to the mean of the prior three sentences, norm jumps, delta-vector size, and raw-block PCA summaries.',
            'These motivate why the feature set does not stop at attention: sometimes the attention view is broad, but the sentence representation still becomes unusually similar to a particular earlier sentence or unusually low-rank internally.',
            'If the model is reusing a specific earlier hidden state, similarity-to-sentence-i-1 or similarity-to-prior should rise alongside a more structured sentence representation.',
            fig=activation_plot['fig'],
            note=activation_plot['note'],
        )
    )

    trace_salience_plot = plot_trace_salience_features(feature_view)
    sections.append(
        build_plot_section(
            'Feature change and trace salience',
            'A compact summary of which feature families spike relative to the previous sentence and which are unusually extreme relative to all earlier sentences in the same trace, now focused on the spike-friendly attention ratios and prior-context concentration metrics.',
            'This motivates the change features and within-trace normalization features in attention_features2.py: the absolute value alone is often less informative than whether that value is locally new or historically extreme.',
            'If the selected boundary is special, several rows should light up in the delta heatmap, the percentile heatmap, or both, within a restricted layer range.',
            fig=trace_salience_plot['fig'],
            note=trace_salience_plot['note'],
        )
    )

    prior_similarity_heatmap_plot = plot_prior_sentence_similarity_heatmap(residual_view['prior_similarity_df'])
    sections.append(
        build_plot_section(
            'Prior-sentence similarity heatmap',
            'A layer-by-prior-sentence view of how similar the selected sentence residual state is to earlier sentence states.',
            'This is the residual-space companion to the sentence-offset attention plots. It asks whether the selected sentence representation itself starts to resemble a particular earlier sentence, not just attend to it.',
            'If a specific earlier thought is being reactivated, one or a few prior sentences should stand out in a limited layer band rather than the whole column pattern staying diffuse.',
            fig=prior_similarity_heatmap_plot['fig'],
            note=prior_similarity_heatmap_plot['note'],
        )
    )

    sentence_entropy_plot = plot_sentence_entropy_summary(attention_entropy_df, prior_similarity_entropy_df)
    sections.append(
        build_plot_section(
            'Sentence concentration summary',
            'A cross-layer summary of how concentrated the selected sentence is over the selected sentence itself plus earlier sentences, including entropy, top-sentence share, effective sentence count, and the distance to the winning sentence.',
            'This is a high-level summary of the sentence-winner heatmaps and is especially useful for motivating the concentration-style features in attention_features2.py.',
            'If deception relies on recalling a specific earlier plan or claim, concentration should increase around informative layers and the effective number of active sentences should fall.',
            fig=sentence_entropy_plot['fig'],
            note=sentence_entropy_plot['note'],
        )
    )

    prior_winner_plot = plot_prior_sentence_winner_trace(attention_entropy_df, prior_similarity_entropy_df)
    sections.append(
        build_plot_section(
            'Sentence winner trace',
            'Across layers, which sentence wins and how strongly it beats the runner-up under both the attention and residual-similarity views.',
            'This gives a cleaner answer to the question of whether the model keeps coming back to the same sentence, including the selected sentence itself, which can get lost in full heatmaps.',
            'If one earlier thought is actually driving the deceptive move, the winning sentence should stabilize across layers and its margin should increase near the important layers.',
            fig=prior_winner_plot['fig'],
            note=prior_winner_plot['note'],
        )
    )

    prior_similarity_plot = plot_prior_sentence_similarity(focus_prior_sentence_df, focus_layer=effective_focus_layer)
    sections.append(
        build_plot_section(
            'Prior-sentence residual similarity',
            'The cosine similarity between the selected sentence residual state and earlier sentence residual states at the chosen layer.',
            'This helps you see whether the current deceptive-looking sentence is reusing or reactivating an earlier thought or claim from the reasoning trace.',
            'If a concrete earlier sentence is being reused, the selected layer should show a visible standout sentence instead of a nearly flat profile over prior sentences.',
            fig=prior_similarity_plot['fig'],
            note=prior_similarity_plot['note'],
        )
    )

    return {
        'env_name': env_name,
        'dataset_root': env_cfg['dataset_root'],
        'example_id': boundary_row['example_id'],
        'boundary_id': boundary_id,
        'sections': sections,
    }


## 5. Default analysis

The cell below only initializes the default explorer state. It does **not** auto-render plots, which helps avoid duplicate outputs.


In [11]:
DEFAULT_FOCUS_LAYER = max(0, int(ENV_CONFIGS[DEFAULT_ENV]['num_layers']) - 1)


## 6. Interactive explorer

Use the controls below to choose a source environment and one spike example. The left panel shows the spike sentence boundary, and the right panel is automatically set to the immediately previous sentence in the same example. The query is fixed to the last token of the selected sentence so the attention and activation traces are anchored to the sentence end token. Click **Run comparison** once after making your selections.


In [12]:

try:
    import ipywidgets as widgets
except Exception as exc:
    raise ImportError('ipywidgets is required for the spike-vs-previous explorer. Install/enable it and rerun this cell.') from exc

clear_output(wait=True)

SIDE_PANEL_SPECS = {
    'spike': {
        'label': 'Spike sentence',
        'accent_color': '#db4437',
    },
    'prior': {
        'label': 'Previous sentence',
        'accent_color': '#4e79a7',
    },
}
SIDE_PANEL_ORDER = ['spike', 'prior']

for stale_name in [
    'env_widget',
    'query_mode_widget',
    'focus_group_widget',
    'focus_layer_widget',
    'run_button',
    'analysis_box',
    'header_box',
    'controls',
    'example_widget',
    'boundary_widget',
    'previous_boundary_box',
]:
    stale_obj = globals().get(stale_name)
    if stale_obj is not None and hasattr(stale_obj, 'close'):
        try:
            stale_obj.close()
        except Exception:
            pass

env_widget = widgets.Dropdown(options=[(env_name, env_name) for env_name in ENV_ORDER], value=DEFAULT_ENV, description='Env', layout=widgets.Layout(width='55%'))
query_mode_widget = widgets.Dropdown(options=[('Last token of selected sentence (fixed)', 'last_token_selected_sentence')], value='last_token_selected_sentence', description='Query', disabled=True, layout=widgets.Layout(width='65%'))
focus_group_widget = widgets.Dropdown(options=[], description='Group', layout=widgets.Layout(width='65%'))
focus_layer_widget = widgets.IntSlider(value=DEFAULT_FOCUS_LAYER, min=0, max=int(ENV_CONFIGS[DEFAULT_ENV]['num_layers']) - 1, step=1, description='Layer', continuous_update=False, layout=widgets.Layout(width='65%'))
run_button = widgets.Button(description='Run comparison', button_style='primary', icon='play')
header_box = widgets.HTML()
analysis_box = widgets.VBox([], layout=widgets.Layout(width='100%'))
example_widget = widgets.Dropdown(options=[], description='Spike example', layout=widgets.Layout(width='70%'))
boundary_widget = widgets.Dropdown(options=[], description='Spike boundary', layout=widgets.Layout(width='70%'))
previous_boundary_box = widgets.HTML(layout=widgets.Layout(width='70%'))


def _refresh_focus_controls(*_args):
    cfg = get_env_config(env_widget.value)
    group_options = [(group_name, group_name) for group_name in cfg['attention_group_order']]
    focus_group_widget.options = group_options
    group_values = [value for _, value in group_options]
    if cfg['default_focus_group'] in group_values:
        focus_group_widget.value = cfg['default_focus_group']
    elif group_values:
        focus_group_widget.value = group_values[0]
    focus_layer_widget.max = int(cfg['num_layers']) - 1
    if focus_layer_widget.value > focus_layer_widget.max:
        focus_layer_widget.value = focus_layer_widget.max


def _spike_example_df():
    return get_example_subset_df(env_widget.value, 'top_spike_examples').reset_index(drop=True)


def _spike_example_options():
    subset_df = _spike_example_df()
    return [(row['option_label'], int(row_idx)) for row_idx, row in subset_df.iterrows()]


def _selected_example_id():
    subset_df = _spike_example_df()
    if subset_df.empty or example_widget.value is None:
        return None
    selected_row_idx = max(0, min(int(example_widget.value), len(subset_df) - 1))
    return str(subset_df.iloc[selected_row_idx]['example_id'])


def _boundary_rows_for_selected_example():
    example_id = _selected_example_id()
    if example_id is None:
        return pd.DataFrame()
    rows = get_boundary_rows_for_example(env_widget.value, example_id).reset_index(drop=True)
    if 'sentence_idx' in rows.columns:
        rows = rows.sort_values('sentence_idx').reset_index(drop=True)
    return rows


def _boundary_options():
    rows = _boundary_rows_for_selected_example()
    return [(row['boundary_option_label'], int(row_idx)) for row_idx, row in rows.iterrows()]


def _selected_spike_boundary_row():
    rows = _boundary_rows_for_selected_example()
    if rows.empty or boundary_widget.value is None:
        return None
    selected_row_idx = max(0, min(int(boundary_widget.value), len(rows) - 1))
    return rows.iloc[selected_row_idx]


def _selected_prior_boundary_row():
    rows = _boundary_rows_for_selected_example()
    spike_row = _selected_spike_boundary_row()
    if rows.empty or spike_row is None:
        return None
    selected_row_idx = int(spike_row.name)
    if selected_row_idx <= 0:
        return None
    return rows.iloc[selected_row_idx - 1]


def _selected_boundary_id(panel_key):
    row = _selected_spike_boundary_row() if panel_key == 'spike' else _selected_prior_boundary_row()
    if row is None:
        return None
    return str(row['boundary_id'])


def _selected_boundary_row(panel_key):
    return _selected_spike_boundary_row() if panel_key == 'spike' else _selected_prior_boundary_row()


def _preferred_boundary_index():
    rows = _boundary_rows_for_selected_example()
    if rows.empty:
        return None
    default_boundary_id = default_boundary_id_for_example(
        env_widget.value,
        _selected_example_id(),
        subset_name='top_spike_examples',
    )
    matching_rows = [idx for idx, row in rows.iterrows() if str(row['boundary_id']) == str(default_boundary_id)]
    if matching_rows and matching_rows[0] > 0:
        return int(matching_rows[0])
    if len(rows) > 1:
        return 1
    return 0


def _panel_summary_html(panel_key):
    panel_spec = SIDE_PANEL_SPECS[panel_key]
    example_id = _selected_example_id()
    spike_row = _selected_spike_boundary_row()
    boundary_row = _selected_boundary_row(panel_key)
    card_style = (
        f"flex: 1 1 340px; min-width: 340px; border: 1px solid #ddd; "
        f"border-left: 6px solid {panel_spec['accent_color']}; background: white; padding: 10px 12px;"
    )
    if boundary_row is None:
        if panel_key == 'prior' and spike_row is not None:
            return (
                f"<div style='{card_style}'>"
                f"<div><strong>{html.escape(panel_spec['label'])}</strong></div>"
                "<div style='color: #b00020; margin-top: 6px;'>No sentence - 1 comparison is available for the selected spike boundary.</div>"
                "</div>"
            )
        return (
            f"<div style='{card_style}'>"
            f"<div><strong>{html.escape(panel_spec['label'])}</strong></div>"
            "<div style='color: #666; margin-top: 6px;'>No boundary is currently selected for this side.</div>"
            "</div>"
        )
    boundary_strength = describe_boundary_strength(float(boundary_row['delta']))
    compared_to = ''
    if panel_key == 'prior' and spike_row is not None:
        compared_to = f"<div><strong>Compared to spike:</strong> {html.escape(str(spike_row['boundary_id']))}</div>"
    return (
        f"<div style='{card_style}'>"
        f"<div style='margin-bottom: 6px;'><strong>{html.escape(panel_spec['label'])}</strong></div>"
        f"<div><strong>Example:</strong> {html.escape(str(example_id or 'n/a'))}</div>"
        f"<div><strong>Boundary:</strong> {html.escape(str(boundary_row['boundary_id']))}</div>"
        f"{compared_to}"
        f"<div><strong>Sentence:</strong> {html.escape(_preview_text(str(boundary_row['sentence_text'] or ''), 120))}</div>"
        f"<div><strong>Jump:</strong> {html.escape('Δ {:+0.3f}'.format(float(boundary_row['delta'])))}</div>"
        f"<div><strong>Strength:</strong> {html.escape(boundary_strength)}</div>"
        "</div>"
    )


def _update_previous_boundary_note():
    spike_row = _selected_spike_boundary_row()
    prior_row = _selected_prior_boundary_row()
    if spike_row is None:
        previous_boundary_box.value = (
            "<div style='margin: 8px 0 12px 0; padding: 10px 12px; border: 1px solid #ddd; background: #fff;'>"
            "Select a spike example and boundary to see the sentence - 1 comparison."
            "</div>"
        )
        run_button.disabled = True
        return
    if prior_row is None:
        previous_boundary_box.value = (
            "<div style='margin: 8px 0 12px 0; padding: 10px 12px; border: 1px solid #ddd; background: #fff; color: #b00020;'>"
            "The selected spike boundary is sentence 0 in this example, so there is no sentence - 1 comparison available."
            "</div>"
        )
        run_button.disabled = True
        return
    previous_boundary_box.value = (
        "<div style='margin: 8px 0 12px 0; padding: 10px 12px; border: 1px solid #ddd; background: #fff;'>"
        f"<div><strong>Spike boundary:</strong> {html.escape(str(spike_row['boundary_id']))}</div>"
        f"<div><strong>Previous boundary:</strong> {html.escape(str(prior_row['boundary_id']))}</div>"
        f"<div><strong>Previous sentence:</strong> {html.escape(_preview_text(str(prior_row['sentence_text'] or ''), 140))}</div>"
        "</div>"
    )
    run_button.disabled = False


def _update_header():
    dataset_root = get_env_config(env_widget.value)['dataset_root']
    panel_cards_html = ''.join(_panel_summary_html(panel_key) for panel_key in SIDE_PANEL_ORDER)
    header_box.value = (
        "<div style='margin: 12px 0 18px 0; padding: 12px 14px; border: 1px solid #ddd; background: #fafafa;'>"
        f"<div><strong>Env:</strong> {html.escape(str(env_widget.value))}</div>"
        f"<div><strong>Dataset root:</strong> {html.escape(str(dataset_root))}</div>"
        "<div><strong>Comparison:</strong> spike sentence on the left, previous sentence from the same example on the right</div>"
        "<div style='display: flex; gap: 12px; flex-wrap: wrap; margin-top: 12px;'>"
        f"{panel_cards_html}"
        "</div>"
        "</div>"
    )


def _refresh_example_options(reset_example=False):
    options = _spike_example_options()
    example_widget.options = options
    if not options:
        example_widget.value = None
        boundary_widget.options = []
        boundary_widget.value = None
        _update_previous_boundary_note()
        _update_header()
        return
    candidate_values = [value for _, value in options]
    if reset_example or example_widget.value not in candidate_values:
        example_widget.value = candidate_values[0]
    _refresh_boundary_options(reset_boundary=True)


def _refresh_boundary_options(reset_boundary=False):
    rows = _boundary_rows_for_selected_example()
    boundary_widget.options = [(row['boundary_option_label'], int(row_idx)) for row_idx, row in rows.iterrows()]
    if rows.empty:
        boundary_widget.value = None
        _update_previous_boundary_note()
        _update_header()
        return
    candidate_values = [value for _, value in boundary_widget.options]
    if reset_boundary or boundary_widget.value not in candidate_values:
        preferred_idx = _preferred_boundary_index()
        if preferred_idx is None or preferred_idx not in candidate_values:
            preferred_idx = candidate_values[0]
        boundary_widget.value = int(preferred_idx)
    _update_previous_boundary_note()
    _update_header()


def _build_plot_card(section, *, panel_key):
    panel_spec = SIDE_PANEL_SPECS[panel_key]
    children = [
        widgets.HTML(
            f"<div style='margin: 0 0 10px 0; padding: 8px 10px; border-left: 4px solid {panel_spec['accent_color']}; background: #f8f8f8;'>"
            f"<strong>{html.escape(panel_spec['label'])}</strong>"
            "</div>"
        ),
    ]
    if section is None:
        children.append(widgets.HTML("<p style='margin: 0; color: #666;'>No section was produced for this selection.</p>"))
        return widgets.VBox(children, layout=widgets.Layout(width='49%', border='1px solid #eee', padding='10px'))
    if section.get('note'):
        children.append(widgets.HTML(f"<p style='margin: 0 0 10px 0; color: #444;'><strong>Note:</strong> {html.escape(section['note'])}</p>"))
    if section.get('image_png_bytes') is not None:
        children.append(
            widgets.Image(
                value=section['image_png_bytes'],
                format='png',
                layout=widgets.Layout(width='100%'),
            )
        )
    else:
        children.append(widgets.HTML("<p style='margin: 0; color: #666;'>No figure was available for this section.</p>"))
    return widgets.VBox(children, layout=widgets.Layout(width='49%', border='1px solid #eee', padding='10px'))


def _build_comparison_section_widget(section_map):
    reference_section = next((section_map.get(panel_key) for panel_key in SIDE_PANEL_ORDER if section_map.get(panel_key) is not None), None)
    if reference_section is None:
        return widgets.HTML('<p>No comparison sections were available.</p>')
    intro_html = (
        f"<h3 style='margin: 0 0 8px 0;'>{html.escape(reference_section['title'])}</h3>"
        f"<p style='margin: 0 0 6px 0;'><strong>What it shows:</strong> {html.escape(reference_section['what_it_shows'])}</p>"
        f"<p style='margin: 0 0 6px 0;'><strong>Why it is interesting for your problem:</strong> {html.escape(reference_section['why_interesting'])}</p>"
        f"<p style='margin: 0 0 10px 0;'><strong>Hypothesis / expected pattern:</strong> {html.escape(reference_section['hypothesis'])}</p>"
    )
    card_widgets = [_build_plot_card(section_map.get(panel_key), panel_key=panel_key) for panel_key in SIDE_PANEL_ORDER]
    return widgets.VBox(
        [
            widgets.HTML(intro_html),
            widgets.HBox(
                card_widgets,
                layout=widgets.Layout(width='100%', align_items='flex-start', justify_content='space-between', flex_flow='row wrap'),
            ),
        ],
        layout=widgets.Layout(width='100%', margin='0 0 28px 0'),
    )


def _run_selected(_=None):
    _update_header()
    example_id = _selected_example_id()
    spike_boundary_id = _selected_boundary_id('spike')
    prior_boundary_id = _selected_boundary_id('prior')
    if example_id is None:
        analysis_box.children = (widgets.HTML('<p>No spike example is currently selected.</p>'),)
        return
    if spike_boundary_id is None:
        analysis_box.children = (widgets.HTML('<p>No spike boundary is currently selected.</p>'),)
        return
    if prior_boundary_id is None:
        analysis_box.children = (widgets.HTML('<p>The selected spike boundary has no sentence - 1 comparison. Pick a different spike boundary.</p>'),)
        return
    reports = {
        'spike': build_boundary_analysis_report(
            env_widget.value,
            spike_boundary_id,
            query_mode=query_mode_widget.value,
            focus_group=focus_group_widget.value,
            focus_layer=focus_layer_widget.value,
        ),
        'prior': build_boundary_analysis_report(
            env_widget.value,
            prior_boundary_id,
            query_mode=query_mode_widget.value,
            focus_group=focus_group_widget.value,
            focus_layer=focus_layer_widget.value,
        ),
    }
    section_count = max(len(reports[panel_key]['sections']) for panel_key in SIDE_PANEL_ORDER)
    children = [
        widgets.HTML(
            "<p style='margin: 0 0 18px 0; color: #444;'>"
            "The left card is the spike sentence boundary and the right card is the immediately previous sentence in the same example. Both panels are anchored to the last token of the selected sentence."
            "</p>"
        )
    ]
    for section_idx in range(section_count):
        section_map = {
            panel_key: (reports[panel_key]['sections'][section_idx] if section_idx < len(reports[panel_key]['sections']) else None)
            for panel_key in SIDE_PANEL_ORDER
        }
        children.append(_build_comparison_section_widget(section_map))
    analysis_box.children = tuple(children)


def _on_env_change(change):
    _refresh_focus_controls()
    _refresh_example_options(reset_example=True)


def _on_example_change(change):
    _refresh_boundary_options(reset_boundary=True)


def _on_boundary_change(change):
    _update_previous_boundary_note()
    _update_header()


env_widget.observe(_on_env_change, names='value')
example_widget.observe(_on_example_change, names='value')
boundary_widget.observe(_on_boundary_change, names='value')
run_button.on_click(_run_selected)

_refresh_focus_controls()
_refresh_example_options(reset_example=True)
_update_header()

controls = widgets.VBox(
    [
        env_widget,
        widgets.HTML(
            "<div style='margin: 0 0 8px 0; color: #444;'>"
            "Pick one spike example and one spike boundary. The right panel is automatically set to the previous sentence in the same example."
            "</div>"
        ),
        example_widget,
        boundary_widget,
        previous_boundary_box,
        query_mode_widget,
        focus_group_widget,
        focus_layer_widget,
        run_button,
    ]
)
display(widgets.VBox([controls, header_box, analysis_box]))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]